# 🚀 PySpark Production Data Engineering Masterclass: 10-Day Zero-to-Hero
> **Enterprise Data Ingestion, Medallion Architecture, Performance Tuning, PostgreSQL JDBC & Custom DAG Orchestration**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Architecture](https://img.shields.io/badge/Architecture-Medallion%20Lakehouse-008080?style=for-the-badge)](https://www.databricks.com/glossary/medallion-architecture)
[![Cloud Database](https://img.shields.io/badge/Cloud%20Database-Neon%20Postgres-00E599?style=for-the-badge&logo=postgresql&logoColor=white)](https://neon.tech)
[![Orchestration](https://img.shields.io/badge/Orchestrator-Custom%20DAG%20with%20Retries-FF6F00?style=for-the-badge&logo=apacheairflow&logoColor=white)](https://airflow.apache.org/)

---

## 📖 Curriculum & Reading Roadmap

Welcome to the **10-Day PySpark Production Data Engineering Masterclass**! This curriculum takes you step-by-step from raw file ingestion and Medallion Lakehouse patterns to advanced analytical window functions, cluster performance optimization, cloud database JDBC connectivity, and building your own fault-tolerant DAG orchestrator.

```
                     ┌────────────────────────────────────────────────────────┐
                     │          10-DAY PRODUCTION ENGINEERING TRACK           │
                     └──────────────────────────┬─────────────────────────────┘
                                                │
       ┌───────────────────┬────────────────────┼───────────────────┬────────────────────┐
       ▼                   ▼                    ▼                   ▼                    ▼
   [Day 01-03]         [Day 04]             [Day 05-06]         [Day 07-08]          [Day 09-10]
  Medallion &         Performance          Joins, Window       Modular ETL &        Cloud Postgres &
  Incremental        Optimization &        Functions &        Production SLAs      Custom DAG Engine
   Ingestion          Partitioning           Modeling           w/ Testing             w/ Retries
```

| Day | Module Title | Core Focus & Techniques | Output Layer |
|:---:|:---|:---|:---:|
| **[Day 01](#day-01-medallion-architecture-fundamentals-bronze-silver--gold-ingestion)** | Medallion Architecture Fundamentals | Explicit `StructType` Schema, CSV Ingestion, Null handling, Derived Revenue | Bronze, Silver & Gold Parquet |
| **[Day 02](#day-02-production-data-quality-quarantine--validation-framework)** | Production Data Quality Framework | Rejection rules, Quarantine / Dead-Letter Queue (`rejected_orders`), Audit report | Quarantined Silver & Reject tables |
| **[Day 03](#day-03-incremental-ingestion-watermarking--metadata-tracking)** | Incremental Ingestion & Metadata Tracking | Change detection, watermarking (`ingestion_timestamp`), metadata state audit table | Incremental Appended Parquet |
| **[Day 04](#day-04-spark-performance-optimization-partitioning--broadcast-joins)** | Spark Performance & Partitioning | `repartition()` vs `coalesce()`, `partitionBy()`, pruning, broadcast joins, `.explain()` | Optimized Storage & Memory |
| **[Day 05](#day-05-advanced-transformations-multi-table-joins--data-modeling)** | Advanced Joins & Data Modeling | Inner/Left/Right/Full joins, column aliasing, `when/otherwise`, multi-dimensional metrics | Curated Business Mart |
| **[Day 06](#day-06-analytical-window-functions-deep-dive)** | Analytical Window Functions Deep Dive | `row_number()`, `rank()`, `dense_rank()`, `lag()`, `lead()`, running totals, moving average | Customer Time-Series Analytics |
| **[Day 07](#day-07-modular-production-pipeline-architecture--reusable-etl-functions)** | Modular Pipeline Architecture | Decoupled ETL functions (`ingest`, `validate`, `dedup`, `transform`, `write`), modular driver | End-to-End Modular Pipeline |
| **[Day 08](#day-08-enterprise-pipeline-engineering-logging--automated-testing)** | Enterprise Logging & Automated Testing | Central config, file logging, exception handling, rejection thresholds, unit assertions | Enterprise Grade Pipeline |
| **[Day 09](#day-09-cloud-database-connectivity--neon-postgresql-jdbc-lakehouse-ingestion)** | Cloud PostgreSQL JDBC Ingestion | Neon PostgreSQL JDBC, `psycopg2` + Spark bridge, hybrid lakehouse aggregation | Cloud Lakehouse Parquet |
| **[Day 10](#day-10-custom-dag-pipeline-orchestrator-with-retries--fault-tolerance)** | Custom DAG Orchestrator with Retries | Task registration, retry engine, dependency execution, state tracking, failure simulation | Production Orchestration Engine |

---


## 📦 Day 01: Medallion Architecture Fundamentals (Bronze, Silver & Gold Ingestion)
### 🎯 Learning Objectives:
1. Initialize a production-ready **SparkSession** with proper Hadoop & Windows compatibility (`winutils`).
2. Implement **Explicit Schema Enforcement** using `StructType` and `StructField` to prevent corrupt data loading.
3. Establish the **Medallion Architecture**:
   - **Bronze Layer**: Raw, immutable, append-only ingested Parquet data.
   - **Silver Layer**: Cleansed, typed, deduplicated, null-handled data with calculated business fields.
   - **Gold Layer**: Aggregated, query-ready analytical marts (by category, by city).
4. Persist each layer efficiently using Apache Parquet columnar storage format.
---


%pip install winutils-pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
print("PySpark imported successfully")

PySpark imported successfully


In [3]:
import os
from pyspark.sql import SparkSession

# 1. Configure Hadoop environment with downloaded winutils.exe & hadoop.dll
dummy_hadoop_dir = os.path.abspath("./hadoop_dummy")
os.makedirs(os.path.join(dummy_hadoop_dir, "bin"), exist_ok=True)
os.environ["HADOOP_HOME"] = dummy_hadoop_dir
os.environ["PATH"] = os.path.join(dummy_hadoop_dir, "bin") + ";" + os.environ.get("PATH", "")

# 2. Build SparkSession with RawLocalFileSystem configured
spark = SparkSession.builder \
    .appName("BronzePipeline") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()


In [4]:
spark=(
    SparkSession.builder
    .appName("EcommerceDataEngineering")
    .master("local[*]")
    .getOrCreate()
)
spark

In [5]:
import pandas as pd

In [6]:
data = {
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1008, 1009
    ],

    "customer_id": [
        101, 102, 103, 101, 104,
        105, 106, 102, 102, 107
    ],

    "customer_name": [
        "Rahul", "Amit", "Sneha", "Rahul", "Priya",
        "Arjun", "Riya", "Amit", "Amit", "Karan"
    ],

    "city": [
        "Kolkata", "Delhi", "Mumbai", "Kolkata", "Pune",
        "Bangalore", "Chennai", "Delhi", "Delhi", None
    ],

    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor", "Laptop",
        "Mouse", "Keyboard", "Laptop", "Laptop", "Monitor"
    ],

    "category": [
        "Electronics", "Accessories", "Accessories", "Electronics",
        "Electronics", "Accessories", "Accessories", "Electronics",
        "Electronics", "Electronics"
    ],

    "quantity": [
        1, 2, 1, 1, 2,
        3, 2, 1, 1, None
    ],

    "price": [
        55000, 800, 1500, 12000, 55000,
        800, 1500, 55000, 55000, 12000
    ],

    "order_date": [
        "2026-09-01",
        "2026-09-02",
        "2026-09-02",
        "2026-09-03",
        "2026-09-04",
        "2026-09-05",
        "2026-09-06",
        "2026-09-07",
        "2026-09-07",
        "2026-09-08"
    ]
}

pdf = pd.DataFrame(data)

pdf

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,1001,101,Rahul,Kolkata,Laptop,Electronics,1.0,55000,2026-09-01
1,1002,102,Amit,Delhi,Mouse,Accessories,2.0,800,2026-09-02
2,1003,103,Sneha,Mumbai,Keyboard,Accessories,1.0,1500,2026-09-02
3,1004,101,Rahul,Kolkata,Monitor,Electronics,1.0,12000,2026-09-03
4,1005,104,Priya,Pune,Laptop,Electronics,2.0,55000,2026-09-04
5,1006,105,Arjun,Bangalore,Mouse,Accessories,3.0,800,2026-09-05
6,1007,106,Riya,Chennai,Keyboard,Accessories,2.0,1500,2026-09-06
7,1008,102,Amit,Delhi,Laptop,Electronics,1.0,55000,2026-09-07
8,1008,102,Amit,Delhi,Laptop,Electronics,1.0,55000,2026-09-07
9,1009,107,Karan,None,Monitor,Electronics,NaN,12000,2026-09-08


In [7]:
pdf.to_csv(
    "orders.csv",
    index=False
)
print("CSV created successfully")

CSV created successfully


In [8]:
orders_df=(
    spark.read
    .option("header",True)
    .option("inferSchema",True)
    .csv("orders.csv")
)

In [9]:
orders_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|     1.0|55000|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|     2.0|  800|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|     1.0| 1500|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|     1.0|12000|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|     2.0|55000|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|     3.0|  800|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|     2.0| 1500|2026-09-06|
|    1008|        102|         Amit|    Delhi|  Laptop|Elect

In [10]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [11]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [12]:
orders_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|     1.0|55000|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|     2.0|  800|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|     1.0| 1500|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|     1.0|12000|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|     2.0|55000|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|     3.0|  800|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|     2.0| 1500|2026-09-06|
|    1008|        102|         Amit|    Delhi|  Laptop|Elect

In [13]:
orders_df = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv("orders.csv")
)

orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: string (nullable = true)



In [14]:
print("Rows:",orders_df.count())
print("Columns:",len(orders_df.columns))

Rows: 10
Columns: 9


In [15]:
orders_df.columns

['order_id',
 'customer_id',
 'customer_name',
 'city',
 'product',
 'category',
 'quantity',
 'price',
 'order_date']

In [16]:
orders_df.show(5,truncate=False)

+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
|order_id|customer_id|customer_name|city   |product |category   |quantity|price  |order_date|
+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
|1001    |101        |Rahul        |Kolkata|Laptop  |Electronics|NULL    |55000.0|2026-09-01|
|1002    |102        |Amit         |Delhi  |Mouse   |Accessories|NULL    |800.0  |2026-09-02|
|1003    |103        |Sneha        |Mumbai |Keyboard|Accessories|NULL    |1500.0 |2026-09-02|
|1004    |101        |Rahul        |Kolkata|Monitor |Electronics|NULL    |12000.0|2026-09-03|
|1005    |104        |Priya        |Pune   |Laptop  |Electronics|NULL    |55000.0|2026-09-04|
+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
only showing top 5 rows


In [17]:
# Checking Null Values
orders_df.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|       0|          0|            0|   1|      0|       0|      10|    0|         0|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+



In [18]:
#Checking Duplicates
print("Total rows:",orders_df.count())
print(
"Distinct rows:",
    orders_df.dropDuplicates().count()
)

Total rows: 10
Distinct rows: 9


In [19]:
orders_df.groupBy("order_id")\
.count()\
.filter(col("count")>1)\
.show()

+--------+-----+
|order_id|count|
+--------+-----+
|    1008|    2|
+--------+-----+



In [20]:
#Bronze Layer
bronze_df=orders_df

In [21]:
bronze_df.write \
    .mode("overwrite") \
    .parquet("bronze_orders")

In [22]:
bronze_df=spark.read.parquet("bronze_orders")
bronze_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-------+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|  price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-------+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|    NULL|55000.0|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|    NULL|  800.0|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|    NULL| 1500.0|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|    NULL|12000.0|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|    NULL|55000.0|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|    NULL|  800.0|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|    NULL| 1500.0|2026-09-06|
|    1008|        102|         Amit|    

In [23]:
#Clean the data
silver_df=bronze_df.dropDuplicates(["order_id"])

In [24]:
silver_df.count()

9

In [25]:
#Handle NULL quantity
silver_df=silver_df.fillna({
    "quantity":1
})

In [26]:
silver_df.select(
    "order_id",
    "quantity"
).show()

+--------+--------+
|order_id|quantity|
+--------+--------+
|    1005|       1|
|    1008|       1|
|    1002|       1|
|    1001|       1|
|    1006|       1|
|    1007|       1|
|    1003|       1|
|    1004|       1|
|    1009|       1|
+--------+--------+



In [27]:
#Handle Null City
silver_df=silver_df.fillna({
    "city":"Unknown"
})

In [28]:
silver_df=silver_df.withColumn(
    "order_date",
    to_date(col("order_date"))
)

In [29]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)



In [30]:
#Calculate revenue
silver_df=silver_df.withColumn(
    "revenue",
    col("quantity")*col("price")
)

In [31]:
silver_df.select(
    "order_id",
    "product",
    "quantity",
    "price",
    "revenue"
).show()

+--------+--------+--------+-------+-------+
|order_id| product|quantity|  price|revenue|
+--------+--------+--------+-------+-------+
|    1001|  Laptop|       1|55000.0|55000.0|
|    1002|   Mouse|       1|  800.0|  800.0|
|    1003|Keyboard|       1| 1500.0| 1500.0|
|    1004| Monitor|       1|12000.0|12000.0|
|    1005|  Laptop|       1|55000.0|55000.0|
|    1006|   Mouse|       1|  800.0|  800.0|
|    1007|Keyboard|       1| 1500.0| 1500.0|
|    1008|  Laptop|       1|55000.0|55000.0|
|    1009| Monitor|       1|12000.0|12000.0|
+--------+--------+--------+-------+-------+



In [32]:
silver_df.write\
.mode("overwrite")\
.parquet("silver_orders")

In [33]:
#Gold dataset
from pyspark.sql.functions import countDistinct, sum
gold_category=(
    silver_df
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)

In [34]:
gold_category.show()

+-----------+-------------+------------+
|   category|total_revenue|total_orders|
+-----------+-------------+------------+
|Electronics|     189000.0|           5|
|Accessories|       4600.0|           4|
+-----------+-------------+------------+



In [35]:
#Gold by City
gold_city = (
    silver_df
    .groupBy("city")
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(desc("total_revenue"))
)

In [36]:
gold_city.show()

+---------+-------------+----------------+
|     city|total_revenue|unique_customers|
+---------+-------------+----------------+
|  Kolkata|      67000.0|               1|
|    Delhi|      55800.0|               1|
|     Pune|      55000.0|               1|
|  Unknown|      12000.0|               1|
|  Chennai|       1500.0|               1|
|   Mumbai|       1500.0|               1|
|Bangalore|        800.0|               1|
+---------+-------------+----------------+



In [37]:
#Save Gold
gold_category.write \
    .mode("overwrite") \
    .parquet("gold_category")

In [38]:
gold_city.write \
    .mode("overwrite") \
    .parquet("gold_city")

## 🛡️ Day 02: Production Data Quality, Quarantine & Validation Framework
### 🎯 Learning Objectives:
1. Design a resilient **Production Data Quality (DQ) Engine** that isolates corrupt records without crashing the ETL pipeline.
2. Formulate explicit validation rules:
   - Mandatory foreign keys (`customer_id is NOT NULL`)
   - Domain bounds (`quantity > 0`, `price > 0`)
   - Timestamp integrity (ISO-8601 parsing & validation)
   - Uniqueness constraints (`order_id` deduplication)
3. Implement a **Quarantine / Dead-Letter Queue (DLQ)** pattern:
   - Tagging failure reasons in a new `rejection_reason` column.
   - Writing pristine records to `silver_orders_day2`.
   - Diverting tainted records to `rejected_orders_day2` for audit.
4. Generate an automated **Data Quality Reconciliation Report** and configure standard Python `logging`.
---


### ⚙️ Step 1: Initialize SparkSession for Day 02: Production Data Quality, Quarantine & Validation Framework


In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day2_DataQualityPipeline")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [40]:
import pandas as pd

In [41]:
data = {
    "order_id": [
        2001,
        2002,
        2003,
        2004,
        2005,
        2006,
        2006,
        2007,
        2008,
        2009
    ],

    "customer_id": [
        101,
        102,
        103,
        None,
        105,
        106,
        106,
        107,
        108,
        109
    ],

    "customer_name": [
        "Rahul",
        "Amit",
        "Sneha",
        "Priya",
        "Arjun",
        "Riya",
        "Riya",
        "Karan",
        "Neha",
        "Sourav"
    ],

    "city": [
        "Kolkata",
        "Delhi",
        "Mumbai",
        "Pune",
        "Bangalore",
        "Chennai",
        "Chennai",
        None,
        "Hyderabad",
        "Kolkata"
    ],

    "product": [
        "Laptop",
        "Mouse",
        "Keyboard",
        "Monitor",
        "Laptop",
        "Mouse",
        "Mouse",
        "Keyboard",
        "Laptop",
        "Monitor"
    ],

    "category": [
        "Electronics",
        "Accessories",
        "Accessories",
        "Electronics",
        "Electronics",
        "Accessories",
        "Accessories",
        "Accessories",
        "Electronics",
        "Electronics"
    ],

    "quantity": [
        1,
        2,
        -1,
        1,
        None,
        3,
        3,
        0,
        2,
        1
    ],

    "price": [
        55000,
        800,
        1500,
        12000,
        55000,
        800,
        800,
        -1500,
        55000,
        None
    ],

    "order_date": [
        "2026-09-20",
        "2026-09-20",
        "2026-09-21",
        "2026-09-21",
        "2026-09-22",
        "2026-09-23",
        "2026-09-23",
        "2026-09-24",
        "INVALID_DATE",
        "2026-09-25"
    ]
}

df = pd.DataFrame(data)

df.to_csv(
    "orders_day2.csv",
    index=False
)

df

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,2001,101.0,Rahul,Kolkata,Laptop,Electronics,1.0,55000.0,2026-09-20
1,2002,102.0,Amit,Delhi,Mouse,Accessories,2.0,800.0,2026-09-20
2,2003,103.0,Sneha,Mumbai,Keyboard,Accessories,-1.0,1500.0,2026-09-21
3,2004,NaN,Priya,Pune,Monitor,Electronics,1.0,12000.0,2026-09-21
4,2005,105.0,Arjun,Bangalore,Laptop,Electronics,NaN,55000.0,2026-09-22
5,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
6,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
7,2007,107.0,Karan,None,Keyboard,Accessories,0.0,-1500.0,2026-09-24
8,2008,108.0,Neha,Hyderabad,Laptop,Electronics,2.0,55000.0,INVALID_DATE
9,2009,109.0,Sourav,Kolkata,Monitor,Electronics,1.0,NaN,2026-09-25


In [42]:
df

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,2001,101.0,Rahul,Kolkata,Laptop,Electronics,1.0,55000.0,2026-09-20
1,2002,102.0,Amit,Delhi,Mouse,Accessories,2.0,800.0,2026-09-20
2,2003,103.0,Sneha,Mumbai,Keyboard,Accessories,-1.0,1500.0,2026-09-21
3,2004,NaN,Priya,Pune,Monitor,Electronics,1.0,12000.0,2026-09-21
4,2005,105.0,Arjun,Bangalore,Laptop,Electronics,NaN,55000.0,2026-09-22
5,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
6,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
7,2007,107.0,Karan,None,Keyboard,Accessories,0.0,-1500.0,2026-09-24
8,2008,108.0,Neha,Hyderabad,Laptop,Electronics,2.0,55000.0,INVALID_DATE
9,2009,109.0,Sourav,Kolkata,Monitor,Electronics,1.0,NaN,2026-09-25


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       10 non-null     int64  
 1   customer_id    9 non-null      float64
 2   customer_name  10 non-null     object 
 3   city           9 non-null      object 
 4   product        10 non-null     object 
 5   category       10 non-null     object 
 6   quantity       9 non-null      float64
 7   price          9 non-null      float64
 8   order_date     10 non-null     object 
dtypes: float64(3), int64(1), object(5)
memory usage: 848.0+ bytes


In [44]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [45]:
StructField("order_id", IntegerType(), False)

StructField('order_id', IntegerType(), False)

In [46]:
raw_df=(
    spark.read
    .option("header",True)
    .schema(order_schema)
    .csv("orders_day2.csv")
)

In [47]:
raw_df.show(truncate=False)

+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|order_id|customer_id|customer_name|city     |product |category   |quantity|price  |order_date  |
+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|2001    |NULL       |Rahul        |Kolkata  |Laptop  |Electronics|NULL    |55000.0|2026-09-20  |
|2002    |NULL       |Amit         |Delhi    |Mouse   |Accessories|NULL    |800.0  |2026-09-20  |
|2003    |NULL       |Sneha        |Mumbai   |Keyboard|Accessories|NULL    |1500.0 |2026-09-21  |
|2004    |NULL       |Priya        |Pune     |Monitor |Electronics|NULL    |12000.0|2026-09-21  |
|2005    |NULL       |Arjun        |Bangalore|Laptop  |Electronics|NULL    |55000.0|2026-09-22  |
|2006    |NULL       |Riya         |Chennai  |Mouse   |Accessories|NULL    |800.0  |2026-09-23  |
|2006    |NULL       |Riya         |Chennai  |Mouse   |Accessories|NULL    |800.0  |2026-09-23  |
|2007    |NULL      

In [48]:
total_records=raw_df.count()

In [49]:
print("Total records:",total_records)

Total records: 10


In [50]:
invalid_customer=raw_df.filter(
    col("customer_id").isNull()
)
invalid_customer.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|  price|  order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|    2001|       NULL|        Rahul|  Kolkata|  Laptop|Electronics|    NULL|55000.0|  2026-09-20|
|    2002|       NULL|         Amit|    Delhi|   Mouse|Accessories|    NULL|  800.0|  2026-09-20|
|    2003|       NULL|        Sneha|   Mumbai|Keyboard|Accessories|    NULL| 1500.0|  2026-09-21|
|    2004|       NULL|        Priya|     Pune| Monitor|Electronics|    NULL|12000.0|  2026-09-21|
|    2005|       NULL|        Arjun|Bangalore|  Laptop|Electronics|    NULL|55000.0|  2026-09-22|
|    2006|       NULL|         Riya|  Chennai|   Mouse|Accessories|    NULL|  800.0|  2026-09-23|
|    2006|       NULL|         Riya|  Chennai|   Mouse|Accessories|    NULL|  800.0|  2026-09-23|
|    2007|       NUL

In [51]:
valid_customer=raw_df.filter(
    col("customer_id").isNotNull()
)
valid_customer.show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+



In [52]:
invalid_quantity=raw_df.filter(
    col("quantity").isNull() |
    (col("quantity")<=0)
)
invalid_quantity.select(
    "order_id",
    "quantity"
).show()

+--------+--------+
|order_id|quantity|
+--------+--------+
|    2001|    NULL|
|    2002|    NULL|
|    2003|    NULL|
|    2004|    NULL|
|    2005|    NULL|
|    2006|    NULL|
|    2006|    NULL|
|    2007|    NULL|
|    2008|    NULL|
|    2009|    NULL|
+--------+--------+



In [53]:
invalid_price=raw_df.filter(
    col("price").isNull() |
    (col("price")<=0)
)
invalid_price.select(
    "order_id",
    "price"
).show()

+--------+-------+
|order_id|  price|
+--------+-------+
|    2007|-1500.0|
|    2009|   NULL|
+--------+-------+



In [54]:
validated_df = raw_df.withColumn(
    "parsed_date",
    try_to_date(col("order_date"), "yyyy-MM-dd")
)
display(validated_df)

DataFrame[order_id: int, customer_id: int, customer_name: string, city: string, product: string, category: string, quantity: int, price: double, order_date: string, parsed_date: date]

In [55]:
validated_df.select(
    "order_id",
    "order_date",
    "parsed_date"
).show()

+--------+------------+-----------+
|order_id|  order_date|parsed_date|
+--------+------------+-----------+
|    2001|  2026-09-20| 2026-09-20|
|    2002|  2026-09-20| 2026-09-20|
|    2003|  2026-09-21| 2026-09-21|
|    2004|  2026-09-21| 2026-09-21|
|    2005|  2026-09-22| 2026-09-22|
|    2006|  2026-09-23| 2026-09-23|
|    2006|  2026-09-23| 2026-09-23|
|    2007|  2026-09-24| 2026-09-24|
|    2008|INVALID_DATE|       NULL|
|    2009|  2026-09-25| 2026-09-25|
+--------+------------+-----------+



In [56]:
invalid_date = validated_df.filter(
    col("parsed_date").isNull()
)

invalid_date.select(
    "order_id",
    "order_date"
).show()

+--------+------------+
|order_id|  order_date|
+--------+------------+
|    2008|INVALID_DATE|
+--------+------------+



In [57]:
duplicate_orders = (
    raw_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_orders.show()

+--------+-----+
|order_id|count|
+--------+-----+
|    2006|    2|
+--------+-----+



In [58]:
validation_condition=(
    col("customer_id").isNull()
    |
    col("quantity").isNull()
    |
    (col("quantity")<=0)
    |
    col("price").isNull()
    |
    (col("price")<=0)
    |
    col("parsed_date").isNull()
)

In [59]:
invalid_df=validated_df.filter(
    validation_condition
)

In [60]:
valid_df=validated_df.filter(
    ~validation_condition
)

In [61]:
print("Total:", validated_df.count())
print("Valid:", valid_df.count())
print("Invalid:", invalid_df.count())

Total: 10
Valid: 0
Invalid: 10


In [62]:
rejected_df = (
    validated_df
    .withColumn(
        "rejection_reason",
        when(
            col("customer_id").isNull(),
            lit("customer_id is NULL")
        )
        .when(
            col("quantity").isNull(),
            lit("quantity is NULL")
        )
        .when(
            col("quantity") <= 0,
            lit("quantity must be greater than 0")
        )
        .when(
            col("price").isNull(),
            lit("price is NULL")
        )
        .when(
            col("price") <= 0,
            lit("price must be greater than 0")
        )
        .when(
            col("parsed_date").isNull(),
            lit("Invalid order date")
        )
    )
)

In [63]:
rejected_df = rejected_df.filter(
    col("rejection_reason").isNotNull()
)

In [64]:
rejected_df.select(
    "order_id",
    "customer_id",
    "quantity",
    "price",
    "order_date",
    "rejection_reason"
).show(truncate=False)

+--------+-----------+--------+-------+------------+-------------------+
|order_id|customer_id|quantity|price  |order_date  |rejection_reason   |
+--------+-----------+--------+-------+------------+-------------------+
|2001    |NULL       |NULL    |55000.0|2026-09-20  |customer_id is NULL|
|2002    |NULL       |NULL    |800.0  |2026-09-20  |customer_id is NULL|
|2003    |NULL       |NULL    |1500.0 |2026-09-21  |customer_id is NULL|
|2004    |NULL       |NULL    |12000.0|2026-09-21  |customer_id is NULL|
|2005    |NULL       |NULL    |55000.0|2026-09-22  |customer_id is NULL|
|2006    |NULL       |NULL    |800.0  |2026-09-23  |customer_id is NULL|
|2006    |NULL       |NULL    |800.0  |2026-09-23  |customer_id is NULL|
|2007    |NULL       |NULL    |-1500.0|2026-09-24  |customer_id is NULL|
|2008    |NULL       |NULL    |55000.0|INVALID_DATE|customer_id is NULL|
|2009    |NULL       |NULL    |NULL   |2026-09-25  |customer_id is NULL|
+--------+-----------+--------+-------+------------

In [65]:
valid_df=(
    validated_df
    .filter(~validation_condition)
)

In [66]:
rejected_df=(
    rejected_df
    .select(
        "*"
    )
)

In [67]:
valid_df = valid_df.dropDuplicates(
    ["order_id"]
)

In [68]:
silver_df = (
    valid_df
    .withColumn(
        "order_date",
        col("parsed_date")
    )
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
    .drop("parsed_date")
)

In [69]:
silver_df.show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|revenue|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+



In [70]:
silver_df.write \
    .mode("overwrite") \
    .parquet("silver_orders_day2")

In [71]:
rejected_df.write \
    .mode("overwrite") \
    .parquet("rejected_orders_day2")

In [72]:
total = raw_df.count()
valid = silver_df.count()
rejected = rejected_df.count()

valid_percentage = (valid / total) * 100 if total > 0 else 0.0
rejected_percentage = (rejected / total) * 100 if total > 0 else 0.0

print("========== DATA QUALITY REPORT ==========")
print("Total records    :", total)
print("Valid records    :", valid)
print("Rejected records :", rejected)
print(f"Valid %          : {valid_percentage:.2f}")
print(f"Rejected %       : {rejected_percentage:.2f}")

========== DATA QUALITY REPORT ==========
Total records    : 10
Valid records    : 0
Rejected records : 10
Valid %          : 0.00
Rejected %       : 100.00


In [73]:
rejected_df.groupBy(
    "rejection_reason"
).count().orderBy(
    desc("count")
).show(truncate=False)

+-------------------+-----+
|rejection_reason   |count|
+-------------------+-----+
|customer_id is NULL|10   |
+-------------------+-----+



In [74]:
rejection_rate = rejected / total

if rejection_rate > 0.30:
    pipeline_status = "FAILED"
else:
    pipeline_status = "SUCCESS"

print("Pipeline status:", pipeline_status)

Pipeline status: FAILED


In [75]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("DataPipeline")

In [76]:
logger.info("Pipeline started")
logger.info(f"Total records received: {total}")
logger.info(f"Valid records: {valid}")
logger.info(f"Rejected records: {rejected}")
logger.info(f"Rejection rate: {rejection_rate:.2%}")
logger.info(f"Pipeline status: {pipeline_status}")

2026-10-08 07:30:47,734 - INFO - Pipeline started
2026-10-08 07:30:47,736 - INFO - Total records received: 10
2026-10-08 07:30:47,739 - INFO - Valid records: 0
2026-10-08 07:30:47,742 - INFO - Rejected records: 10
2026-10-08 07:30:47,745 - INFO - Rejection rate: 100.00%
2026-10-08 07:30:47,748 - INFO - Pipeline status: FAILED


## 🔄 Day 03: Incremental Ingestion, Watermarking & Metadata Tracking
### 🎯 Learning Objectives:
1. Transition from expensive **Full Table Scans** to resilient **Incremental Batch Ingestion**.
2. Build an external **Metadata State Store** (`metadata/processed_files.csv`) to track ingested files.
3. Implement file-arrival detection using path patterns and metadata differential matching.
4. Add pipeline audit watermarks (`ingestion_timestamp`) to trace data lineage.
5. Append new partitions into `bronze_incremental` without re-writing historical data.
6. Cascade incremental updates downstream through Silver and Gold layers.
---


### ⚙️ Step 1: Initialize SparkSession for Day 03: Incremental Ingestion, Watermarking & Metadata Tracking


In [77]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day3_IncrementalIngestion")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [78]:
import os

os.makedirs("source", exist_ok=True)

print("Source directory created")

Source directory created


In [79]:
import pandas as pd

day1_data = {
    "order_id": [3001, 3002, 3003, 3004],
    "customer_id": [101, 102, 103, 104],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "quantity": [1, 2, 1, 1],
    "price": [55000, 800, 1500, 12000],
    "order_date": [
        "2026-09-27",
        "2026-09-27",
        "2026-09-27",
        "2026-09-27"
    ]
}

pd.DataFrame(day1_data).to_csv(
    "source/orders_2026_09_27.csv",
    index=False
)

print("Day 1 file created")

Day 1 file created


In [80]:
day2_data = {
    "order_id": [3005, 3006, 3007],
    "customer_id": [105, 106, 107],
    "product": ["Laptop", "Mouse", "Headphones"],
    "quantity": [2, 3, 1],
    "price": [55000, 800, 2500],
    "order_date": [
        "2026-09-28",
        "2026-09-28",
        "2026-09-28"
    ]
}

pd.DataFrame(day2_data).to_csv(
    "source/orders_2026_09_28.csv",
    index=False
)

print("Day 2 file created")

Day 2 file created


In [81]:
day3_data = {
    "order_id": [3008, 3009, 3010],
    "customer_id": [108, 109, 110],
    "product": ["Keyboard", "Laptop", "Monitor"],
    "quantity": [2, 1, 2],
    "price": [1500, 55000, 12000],
    "order_date": [
        "2026-09-29",
        "2026-09-29",
        "2026-09-29"
    ]
}

pd.DataFrame(day3_data).to_csv(
    "source/orders_2026_09_29.csv",
    index=False
)

print("Day 3 file created")

Day 3 file created


In [82]:
os.listdir("source")

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [83]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [84]:
spark.read.csv("source/*.csv")

DataFrame[_c0: string, _c1: string, _c2: string, _c3: string, _c4: string, _c5: string]

In [85]:
all_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv("source/*.csv")
)

In [86]:
all_orders.show()

+--------+-----------+----------+--------+-------+----------+
|order_id|customer_id|   product|quantity|  price|order_date|
+--------+-----------+----------+--------+-------+----------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|
|    3007|        107|Headphones|       1| 2500.0|2026-09-28|
|    3011|        111|     Mouse|       2|  800.0|2026-09-30|
|    3012|        112|    Laptop|       1|55000.0|2026-09-30|
+--------+-----------+----------+--------+-------+----------+



In [87]:
os.makedirs("metadata", exist_ok=True)

print("Metadata directory created")

Metadata directory created


In [88]:
processed_files = pd.DataFrame({
    "file_name": []
})

processed_files.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [89]:
pd.read_csv("metadata/processed_files.csv")

,file_name


In [90]:
import glob

source_files = glob.glob(
    "source/*.csv"
)

source_files

['source\\orders_2026_09_27.csv',
 'source\\orders_2026_09_28.csv',
 'source\\orders_2026_09_29.csv',
 'source\\orders_2026_09_30.csv']

In [91]:
source_file_names = [
    os.path.basename(file)
    for file in source_files
]

source_file_names

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [92]:
processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

processed_files

set()

In [93]:
new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

new_files

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [94]:
new_file_paths = [
    f"source/{file}"
    for file in new_files
]

new_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv(new_file_paths)
)

In [95]:
new_orders.show()

+--------+-----------+----------+--------+-------+----------+
|order_id|customer_id|   product|quantity|  price|order_date|
+--------+-----------+----------+--------+-------+----------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|
|    3007|        107|Headphones|       1| 2500.0|2026-09-28|
|    3011|        111|     Mouse|       2|  800.0|2026-09-30|
|    3012|        112|    Laptop|       1|55000.0|2026-09-30|
+--------+-----------+----------+--------+-------+----------+



In [96]:
new_orders.count()

12

In [97]:
new_orders = new_orders.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [98]:
new_orders.show()

+--------+-----------+----------+--------+-------+----------+--------------------+
|order_id|customer_id|   product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+----------+--------+-------+----------+--------------------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|2026-10-08 07:30:...|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|2026-10-08 07:30:...|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|2026-10-08 07:30:...|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|2026-10-08 07:30:...|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|2026-10-08 07:30:...|
|   

In [99]:
new_orders.write \
    .mode("append") \
    .parquet("bronze_incremental")

In [100]:
processed_df = pd.DataFrame({
    "file_name": new_files
})

processed_df.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [101]:
pd.read_csv(
    "metadata/processed_files.csv"
)

,file_name
0,orders_2026_09_27.csv
1,orders_2026_09_28.csv
2,orders_2026_09_29.csv
3,orders_2026_09_30.csv


In [102]:
day4_data = {
    "order_id": [3011, 3012],
    "customer_id": [111, 112],
    "product": ["Mouse", "Laptop"],
    "quantity": [2, 1],
    "price": [800, 55000],
    "order_date": [
        "2026-09-30",
        "2026-09-30"
    ]
}

pd.DataFrame(day4_data).to_csv(
    "source/orders_2026_09_30.csv",
    index=False
)

print("New file arrived!")

New file arrived!


In [103]:
source_files = glob.glob(
    "source/*.csv"
)

source_file_names = [
    os.path.basename(file)
    for file in source_files
]

processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

new_files

[]

In [104]:
new_file_paths = [
    f"source/{file}"
    for file in new_files
]

new_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv(new_file_paths)
)

In [105]:
new_orders.show()

+--------+-----------+-------+--------+-----+----------+
|order_id|customer_id|product|quantity|price|order_date|
+--------+-----------+-------+--------+-----+----------+
+--------+-----------+-------+--------+-----+----------+



In [106]:
new_orders = new_orders.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

new_orders.write \
    .mode("append") \
    .parquet("bronze_incremental")

In [107]:
old_processed = pd.read_csv(
    "metadata/processed_files.csv"
)

new_processed = pd.DataFrame({
    "file_name": new_files
})

updated_processed = pd.concat(
    [old_processed, new_processed],
    ignore_index=True
)

updated_processed.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [108]:
pd.read_csv(
    "metadata/processed_files.csv"
)

,file_name
0,orders_2026_09_27.csv
1,orders_2026_09_28.csv
2,orders_2026_09_29.csv
3,orders_2026_09_30.csv


In [109]:
source_files = glob.glob(
    "source/*.csv"
)

source_file_names = [
    os.path.basename(file)
    for file in source_files
]

processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

print("New files:", new_files)

New files: []


In [110]:
bronze_df = spark.read.parquet(
    "bronze_incremental"
)

bronze_df.show()

+--------+-----------+--------+--------+-------+----------+--------------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+--------+--------+-------+----------+--------------------+
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-05 08:03:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-05 08:03:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-05 08:03:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-05 08:03:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-03 07:56:...|
|    3002|        102|   Mou

In [111]:
silver_df = (
    bronze_df
    .withColumn(
        "order_date",
        to_date(col("order_date"))
    )
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
)

In [112]:
gold_df = (
    silver_df
    .groupBy("product")
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy(desc("total_revenue"))
)

In [113]:
gold_df.show()

+----------+--------------+-------------+------------+
|   product|total_quantity|total_revenue|total_orders|
+----------+--------------+-------------+------------+
|    Laptop|            65|    3575000.0|           4|
|   Monitor|            39|     468000.0|           2|
|     Mouse|            91|      72800.0|           3|
|  Keyboard|            39|      58500.0|           2|
|Headphones|            13|      32500.0|           1|
+----------+--------------+-------------+------------+



## ⚡ Day 04: Spark Performance Optimization, Partitioning & Broadcast Joins
### 🎯 Learning Objectives:
1. Understand Apache Spark's execution engine: **Lazy Evaluation**, **Transformations (Narrow vs Wide)**, and **Actions**.
2. Master partition management:
   - `repartition(N)`: Full cluster shuffle, redistributing evenly or scaling up partitions.
   - `coalesce(N)`: Shuffle-free partition reduction for efficient writes.
3. Implement **Physical Directory Partitioning** (`partitionBy("order_date")`) and verify **Partition Pruning** using `.explain()`.
4. Avoid the critical anti-pattern: The **Small Files Problem** caused by high-cardinality partition columns.
5. Eliminate heavy network shuffles using **Broadcast Hash Joins** (`broadcast()`) for dimensional lookups.
6. Manage in-memory cache lifecycle with `.cache()`, `.is_cached`, and `.unpersist()`.
7. Conduct empirical query performance benchmarking across partition topologies.
---


### ⚙️ Step 1: Initialize SparkSession for Day 04: Spark Performance Optimization, Partitioning & Broadcast Joins


In [114]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day4_SparkPerformance")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [115]:
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Default parallelism: 8


In [116]:
bronze_df = spark.read.parquet(
    "bronze_incremental"
)

bronze_df.show()

+--------+-----------+--------+--------+-------+----------+--------------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+--------+--------+-------+----------+--------------------+
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-05 08:03:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-05 08:03:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-05 08:03:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-05 08:03:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-03 07:56:...|
|    3002|        102|   Mou

In [117]:
print("Rows:", bronze_df.count())

Rows: 156


In [118]:
print(
    "Number of partitions:",
    bronze_df.rdd.getNumPartitions()
)

Number of partitions: 8


In [119]:
large_df = (
    bronze_df
    .withColumn(
        "copy_id",
        explode(
            sequence(
                lit(1),
                lit(1000)
            )
        )
    )
)

In [120]:
large_df.count()

156000

In [121]:
print(
    "Partitions:",
    large_df.rdd.getNumPartitions()
)

Partitions: 8


In [122]:
repartitioned_df = large_df.repartition(8)

In [123]:
print(
    "Partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Partitions: 8


In [124]:
coalesced_df = large_df.coalesce(2)

In [125]:
print(
    "Partitions:",
    coalesced_df.rdd.getNumPartitions()
)

Partitions: 2


| Operation | Increase partitions | Decrease partitions | Shuffle |
|---|---:|---:|---|
| `repartition()` | ✅ | ✅ | Usually yes |
| `coalesce()` | ❌ | ✅ | Usually avoids full shuffle |


Need MORE partitions?
        ↓
   repartition()

Need FEWER partitions?
        ↓
    coalesce()


In [128]:
repartitioned_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=2662]
         +- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
            +- *(1) ColumnarToRow
               +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_date:string,in...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(8

In [129]:
coalesced_df.explain()

== Physical Plan ==
Coalesce 2
+- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_date:string,in...




In [130]:
test_df = large_df.filter(
    col("quantity") > 1
)

print("Transformation created")

Transformation created


Has Spark actually processed the data?
No.
Spark uses lazy evaluation.
The actual computation happens when you call an action, such as:


count()
show()
collect()
write


In [133]:
test_df.count()

78000

Transformation
     ↓
Builds execution plan
     ↓
No immediate computation

Action
     ↓
Spark executes plan


Transformations
select()
filter()
withColumn()
groupBy()
join()
repartition()


Actions
count()
show()
collect()
write


In [137]:
filtered_df = large_df.filter(
    col("quantity") > 1
)

In [138]:
filtered_df.count()

78000

In [139]:
filtered_df.groupBy("product").count().show()

+--------+-----+
| product|count|
+--------+-----+
|   Mouse|39000|
|Keyboard|13000|
| Monitor|13000|
|  Laptop|13000|
+--------+-----+



In [140]:
#Cache
filtered_df = large_df.filter(
    col("quantity") > 1
)

filtered_df.cache()

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [141]:
filtered_df.count()

78000

In [142]:
print(filtered_df.is_cached)

True


In [143]:
filtered_df.unpersist()

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [144]:
print(filtered_df.is_cached)

False


#partitionBy()
PART 13 — partitionBy()
This is different from repartition().
Suppose we want our Parquet data physically organized by:
order_date

We can write:
Cell 15
(    bronze_df    .write    .mode("overwrite")    .partitionBy("order_date")    .parquet("partitioned_orders"))


Now inspect your Jupyter folder.
You'll see something conceptually like:
partitioned_orders/
│
├── order_date=2026-09-27/
│     └── part-....parquet
│
├── order_date=2026-09-28/
│     └── part-....parquet
│
├── order_date=2026-09-29/
│     └── part-....parquet
│
└── order_date=2026-09-30/
      └── part-....parquet

🔥 This is a major Data Engineering concept.
PART 14 — Why partition data?
Imagine you have:
5 TB orders

And someone asks:
Give me orders from September 29.

Without partitioning:
5 TB
 ↓
scan lots of data

With date partitioning:
orders/
   │
   ├── date=2026-09-27
   ├── date=2026-09-28
   ├── date=2026-09-29  ← READ THIS
   └── date=2026-09-30

Spark can potentially avoid reading unrelated partitions.
This is called partition pruning.


In [146]:
(
    bronze_df
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .parquet("partitioned_orders")
)

In [147]:
#Test partition pruning
partitioned_df = spark.read.parquet(
    "partitioned_orders"
)

In [148]:
sept_29 = partitioned_df.filter(
    col("order_date") == "2026-09-29"
)

sept_29.show()

+--------+-----------+--------+--------+-------+--------------------+----------+
|order_id|customer_id| product|quantity|  price| ingestion_timestamp|order_date|
+--------+-----------+--------+--------+-------+--------------------+----------+
|    3008|        108|Keyboard|       2| 1500.0|2026-10-05 08:38:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-05 08:38:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-05 08:38:...|2026-09-29|
|    3008|        108|Keyboard|       2| 1500.0|2026-10-01 07:21:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-01 07:21:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-01 07:21:...|2026-09-29|
|    3008|        108|Keyboard|       2| 1500.0|2026-10-07 07:57:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-07 07:57:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-07 07:57:...|2026-09-29|
|    3008|        108|Keyboa

In [149]:
sept_29.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#2116,customer_id#2117,product#2118,quantity#2119,price#2120,ingestion_timestamp#2121,order_date#2122] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/partitioned_orders], PartitionFilters: [isnotnull(order_date#2122), (order_date#2122 = 2026-09-29)], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,ingestion_timestamp:...




PART 16 — Important warning: too many partitions
Partitioning isn't automatically good.
Bad:
country
 ↓
200 countries

Potentially okay.
But imagine:
user_id
 ↓
10 million users

You could end up with an enormous number of tiny files.
That's the small files problem.
For example:
10 million users
      ↓
10 million partitions/files

Terrible idea.
So partition columns should generally have sensible cardinality and align with common filtering patterns.
For our project:
order_date

is a reasonable practice column.


In [151]:
#Broadcast join
products = spark.createDataFrame(
    [
        (1, "Laptop", "Electronics"),
        (2, "Mouse", "Accessories"),
        (3, "Keyboard", "Accessories"),
        (4, "Monitor", "Electronics"),
        (5, "Headphones", "Accessories")
    ],
    [
        "product_id",
        "product",
        "category"
    ]
)

In [152]:
large_df

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [153]:
products

DataFrame[product_id: bigint, product: string, category: string]

In [154]:
orders_with_products = large_df.join(
    broadcast(products),
    on="product",
    how="left"
)

In [155]:
orders_with_products.show()

+-------+--------+-----------+--------+-------+----------+--------------------+-------+----------+-----------+
|product|order_id|customer_id|quantity|  price|order_date| ingestion_timestamp|copy_id|product_id|   category|
+-------+--------+-----------+--------+-------+----------+--------------------+-------+----------+-----------+
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      1|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      2|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      3|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      4|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      5|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      6|         1|Electronics|
|

What does broadcast mean?
Instead of:
Large Data
   │
   ├── Worker 1
   ├── Worker 2
   ├── Worker 3
   └── Worker 4

Small Data
   │
   └── shuffle around

Spark can send the small table to workers:
             Small lookup
             /    |    \
            ↓     ↓     ↓
        Worker Worker Worker
            ↑     ↑     ↑
          Large distributed data

This can avoid an expensive shuffle for the small side of a join.
But don't broadcast huge tables blindly.


In [157]:
orders_with_products.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product#1767, order_id#1765, customer_id#1766, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771, copy_id#1807, product_id#2146L, category#2148]
   +- BroadcastHashJoin [product#1767], [product#2147], LeftOuter, BuildRight, false, false
      :- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
      :  +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_dat

In [158]:
#Create a performance experiment
df_2 = large_df.repartition(2)
df_4 = large_df.repartition(4)
df_8 = large_df.repartition(8)

In [159]:
print("2:", df_2.rdd.getNumPartitions())
print("4:", df_4.rdd.getNumPartitions())
print("8:", df_8.rdd.getNumPartitions())

2: 2
4: 4
8: 8


In [160]:
#Time an operation
import time

In [161]:
start = time.time()

df_2.groupBy("product").count().collect()

end = time.time()

print("Time:", end - start)

Time: 0.7943155765533447


In [162]:
start = time.time()

df_8.groupBy("product").count().collect()

end = time.time()

print("Time:", end - start)

Time: 0.6839978694915771


## 🔗 Day 05: Advanced Transformations, Multi-Table Joins & Data Modeling
### 🎯 Learning Objectives:
1. Model complex relational star schemas in PySpark across Orders, Customers, and Products.
2. Master all relational join strategies: **Inner**, **Left Outer**, **Right Outer**, and **Full Outer**.
3. Resolve ambiguous column name collisions cleanly using DataFrame aliases (`orders.alias("o")`).
4. Implement complex business rules and branching logic using `F.when().otherwise()`.
5. Execute multi-dimensional aggregations combining `groupBy()` with multi-metric `.agg()`.
6. Conduct customer demographic and geographical spend analysis.
---


### ⚙️ Step 1: Initialize SparkSession for Day 05: Advanced Transformations, Multi-Table Joins & Data Modeling


In [163]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day5_Transformations_Joins")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [164]:
customers_data = [
    (101, "Rahul", "Kolkata"),
    (102, "Amit", "Delhi"),
    (103, "Sneha", "Mumbai"),
    (104, "Priya", "Pune"),
    (105, "Arjun", "Bangalore"),
    (106, "Riya", "Chennai"),
    (107, "Karan", "Hyderabad"),
    (108, "Neha", "Kolkata")
]

customers = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city"]
)

customers.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|        101|        Rahul|  Kolkata|
|        102|         Amit|    Delhi|
|        103|        Sneha|   Mumbai|
|        104|        Priya|     Pune|
|        105|        Arjun|Bangalore|
|        106|         Riya|  Chennai|
|        107|        Karan|Hyderabad|
|        108|         Neha|  Kolkata|
+-----------+-------------+---------+



In [165]:
orders_data = [
    (1001, 101, 501, 1, "2026-09-01"),
    (1002, 102, 502, 2, "2026-09-01"),
    (1003, 101, 503, 1, "2026-09-02"),
    (1004, 103, 504, 2, "2026-09-02"),
    (1005, 104, 501, 1, "2026-09-03"),
    (1006, 105, 502, 3, "2026-09-03"),
    (1007, 106, 503, 2, "2026-09-04"),
    (1008, 102, 505, 1, "2026-09-04"),
    (1009, 107, 504, 1, "2026-09-05"),
    (1010, 108, 501, 1, "2026-09-05"),
    (1011, 101, 505, 2, "2026-09-06"),
    (1012, 103, 502, 4, "2026-09-06")
]

orders = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "product_id", "quantity", "order_date"]
)

orders.show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|    1001|        101|       501|       1|2026-09-01|
|    1002|        102|       502|       2|2026-09-01|
|    1003|        101|       503|       1|2026-09-02|
|    1004|        103|       504|       2|2026-09-02|
|    1005|        104|       501|       1|2026-09-03|
|    1006|        105|       502|       3|2026-09-03|
|    1007|        106|       503|       2|2026-09-04|
|    1008|        102|       505|       1|2026-09-04|
|    1009|        107|       504|       1|2026-09-05|
|    1010|        108|       501|       1|2026-09-05|
|    1011|        101|       505|       2|2026-09-06|
|    1012|        103|       502|       4|2026-09-06|
+--------+-----------+----------+--------+----------+



In [166]:
customers.printSchema()
products.printSchema()
orders.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- product_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_date: string (nullable = true)



In [167]:
orders.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity"
).show()

+--------+-----------+----------+--------+
|order_id|customer_id|product_id|quantity|
+--------+-----------+----------+--------+
|    1001|        101|       501|       1|
|    1002|        102|       502|       2|
|    1003|        101|       503|       1|
|    1004|        103|       504|       2|
|    1005|        104|       501|       1|
|    1006|        105|       502|       3|
|    1007|        106|       503|       2|
|    1008|        102|       505|       1|
|    1009|        107|       504|       1|
|    1010|        108|       501|       1|
|    1011|        101|       505|       2|
|    1012|        103|       502|       4|
+--------+-----------+----------+--------+



In [168]:
products_data = [
    (501, "Laptop", "Electronics", 55000),
    (502, "Mouse", "Accessories", 800),
    (503, "Keyboard", "Accessories", 1500),
    (504, "Monitor", "Electronics", 12000),
    (505, "Headphones", "Accessories", 2500)
]

products = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "price"]
)

products.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       501|      Laptop|Electronics|55000|
|       502|       Mouse|Accessories|  800|
|       503|    Keyboard|Accessories| 1500|
|       504|     Monitor|Electronics|12000|
|       505|  Headphones|Accessories| 2500|
+----------+------------+-----------+-----+



In [169]:
order_product = orders.join(
    products,
    on="product_id",
    how="left"
)

order_product.show()
order_product.printSchema()

+----------+--------+-----------+--------+----------+------------+-----------+-----+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|
+----------+--------+-----------+--------+----------+------------+-----------+-----+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|
|       502|    1002|        102|       2|2026-09-01|       Mouse|Accessories|  800|
|       503|    1003|        101|       1|2026-09-02|    Keyboard|Accessories| 1500|
|       504|    1004|        103|       2|2026-09-02|     Monitor|Electronics|12000|
|       502|    1006|        105|       3|2026-09-03|       Mouse|Accessories|  800|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|
|       503|    1007|        106|       2|2026-09-04|    Keyboard|Accessories| 1500|
|       504|    1009|        107|       1|2026-09-05|     Monitor|Electronics|12000|
|       505|    1008|        102|       1|2026-09-04|  Headphones

In [170]:
order_product = order_product.withColumn(
    "revenue",
    col("quantity") * col("price")
)

order_product.show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|revenue|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|  55000|
|       502|    1002|        102|       2|2026-09-01|       Mouse|Accessories|  800|   1600|
|       503|    1003|        101|       1|2026-09-02|    Keyboard|Accessories| 1500|   1500|
|       504|    1004|        103|       2|2026-09-02|     Monitor|Electronics|12000|  24000|
|       502|    1006|        105|       3|2026-09-03|       Mouse|Accessories|  800|   2400|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|  55000|
|       503|    1007|        106|       2|2026-09-04|    Keyboard|Accessories| 1500|   3000|
|       504|    1009|        107|       1|2026-09-05|     Monitor|Elec

In [171]:
#When and Otherwise
order_product = order_product.withColumn(
    "order_value_category",
    when(col("revenue") >= 50000, "High Value")
    .when(col("revenue") >= 10000, "Medium Value")
    .otherwise("Low Value")
)

In [172]:
order_product.select(
    "order_id",
    "revenue",
    "order_value_category"
).show()

+--------+-------+--------------------+
|order_id|revenue|order_value_category|
+--------+-------+--------------------+
|    1001|  55000|          High Value|
|    1002|   1600|           Low Value|
|    1003|   1500|           Low Value|
|    1004|  24000|        Medium Value|
|    1006|   2400|           Low Value|
|    1005|  55000|          High Value|
|    1007|   3000|           Low Value|
|    1009|  12000|        Medium Value|
|    1008|   2500|           Low Value|
|    1010|  55000|          High Value|
|    1012|   3200|           Low Value|
|    1011|   5000|           Low Value|
+--------+-------+--------------------+



In [173]:
#Filter
high_value_orders = order_product.filter(
    col("revenue") >= 50000
)

high_value_orders.show()

+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+
|product_id|order_id|customer_id|quantity|order_date|product_name|   category|price|revenue|order_value_category|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+
|       501|    1001|        101|       1|2026-09-01|      Laptop|Electronics|55000|  55000|          High Value|
|       501|    1005|        104|       1|2026-09-03|      Laptop|Electronics|55000|  55000|          High Value|
|       501|    1010|        108|       1|2026-09-05|      Laptop|Electronics|55000|  55000|          High Value|
+----------+--------+-----------+--------+----------+------------+-----------+-----+-------+--------------------+



In [174]:
full_orders = order_product.join(
    customers,
    on="customer_id",
    how="left"
)

In [175]:
full_orders.show()

+-----------+----------+--------+--------+----------+------------+-----------+-----+-------+--------------------+-------------+---------+
|customer_id|product_id|order_id|quantity|order_date|product_name|   category|price|revenue|order_value_category|customer_name|     city|
+-----------+----------+--------+--------+----------+------------+-----------+-----+-------+--------------------+-------------+---------+
|        107|       504|    1009|       1|2026-09-05|     Monitor|Electronics|12000|  12000|        Medium Value|        Karan|Hyderabad|
|        103|       504|    1004|       2|2026-09-02|     Monitor|Electronics|12000|  24000|        Medium Value|        Sneha|   Mumbai|
|        103|       502|    1012|       4|2026-09-06|       Mouse|Accessories|  800|   3200|           Low Value|        Sneha|   Mumbai|
|        104|       501|    1005|       1|2026-09-03|      Laptop|Electronics|55000|  55000|          High Value|        Priya|     Pune|
|        106|       503|    1007| 

In [176]:
final_orders = full_orders.select(
    "order_id",
    "order_date",
    "customer_id",
    "customer_name",
    "city",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "price",
    "revenue",
    "order_value_category"
)

final_orders.show()

+--------+----------+-----------+-------------+---------+----------+------------+-----------+--------+-----+-------+--------------------+
|order_id|order_date|customer_id|customer_name|     city|product_id|product_name|   category|quantity|price|revenue|order_value_category|
+--------+----------+-----------+-------------+---------+----------+------------+-----------+--------+-----+-------+--------------------+
|    1001|2026-09-01|        101|        Rahul|  Kolkata|       501|      Laptop|Electronics|       1|55000|  55000|          High Value|
|    1002|2026-09-01|        102|         Amit|    Delhi|       502|       Mouse|Accessories|       2|  800|   1600|           Low Value|
|    1003|2026-09-02|        101|        Rahul|  Kolkata|       503|    Keyboard|Accessories|       1| 1500|   1500|           Low Value|
|    1004|2026-09-02|        103|        Sneha|   Mumbai|       504|     Monitor|Electronics|       2|12000|  24000|        Medium Value|
|    1006|2026-09-03|        105| 

In [177]:
#Aggregation
customer_revenue = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_items")
    )
    .orderBy(desc("total_revenue"))
)

customer_revenue.show()

+-----------+-------------+-------------+------------+-----------+
|customer_id|customer_name|total_revenue|total_orders|total_items|
+-----------+-------------+-------------+------------+-----------+
|        101|        Rahul|        61500|           3|          4|
|        104|        Priya|        55000|           1|          1|
|        108|         Neha|        55000|           1|          1|
|        103|        Sneha|        27200|           2|          6|
|        107|        Karan|        12000|           1|          1|
|        102|         Amit|         4100|           2|          3|
|        106|         Riya|         3000|           1|          2|
|        105|        Arjun|         2400|           1|          3|
+-----------+-------------+-------------+------------+-----------+



In [178]:
category_analysis = (
    final_orders
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue"),
        sum("quantity").alias("units_sold"),
        countDistinct("order_id").alias("orders")
    )
    .orderBy(desc("total_revenue"))
)

category_analysis.show()

+-----------+-------------+----------+------+
|   category|total_revenue|units_sold|orders|
+-----------+-------------+----------+------+
|Electronics|       201000|         6|     5|
|Accessories|        19200|        15|     7|
+-----------+-------------+----------+------+



In [179]:
overall_stats = final_orders.agg(
    sum("revenue").alias("total_revenue"),
    avg("revenue").alias("average_order_value"),
    min("revenue").alias("minimum_order"),
    max("revenue").alias("maximum_order"),
    countDistinct("order_id").alias("total_orders")
)

overall_stats.show()

+-------------+-------------------+-------------+-------------+------------+
|total_revenue|average_order_value|minimum_order|maximum_order|total_orders|
+-------------+-------------------+-------------+-------------+------------+
|       220200|            18350.0|         1500|        55000|          12|
+-------------+-------------------+-------------+-------------+------------+



In [180]:
#Conditional aggregation
final_orders.agg(
    count(
        when(
            col("order_value_category") == "High Value",
            True
        )
    ).alias("high_value_orders"),

    count(
        when(
            col("order_value_category") == "Medium Value",
            True
        )
    ).alias("medium_value_orders"),

    count(
        when(
            col("order_value_category") == "Low Value",
            True
        )
    ).alias("low_value_orders")
).show()

+-----------------+-------------------+----------------+
|high_value_orders|medium_value_orders|low_value_orders|
+-----------------+-------------------+----------------+
|                3|                  2|               7|
+-----------------+-------------------+----------------+



In [181]:
#City-level analysis
city_analysis = (
    final_orders
    .groupBy("city")
    .agg(
        sum("revenue").alias("revenue"),
        countDistinct("customer_id").alias("customers"),
        countDistinct("order_id").alias("orders")
    )
    .orderBy(desc("revenue"))
)

city_analysis.show()

+---------+-------+---------+------+
|     city|revenue|customers|orders|
+---------+-------+---------+------+
|  Kolkata| 116500|        2|     4|
|     Pune|  55000|        1|     1|
|   Mumbai|  27200|        1|     2|
|Hyderabad|  12000|        1|     1|
|    Delhi|   4100|        1|     2|
|  Chennai|   3000|        1|     1|
|Bangalore|   2400|        1|     1|
+---------+-------+---------+------+



In [182]:
customer_products = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        collect_list("product_name").alias("products")
    )
)

customer_products.show(truncate=False)

+-----------+-------------+------------------------------+
|customer_id|customer_name|products                      |
+-----------+-------------+------------------------------+
|102        |Amit         |[Mouse, Headphones]           |
|106        |Riya         |[Keyboard]                    |
|105        |Arjun        |[Mouse]                       |
|103        |Sneha        |[Monitor, Mouse]              |
|101        |Rahul        |[Laptop, Keyboard, Headphones]|
|104        |Priya        |[Laptop]                      |
|107        |Karan        |[Monitor]                     |
|108        |Neha         |[Laptop]                      |
+-----------+-------------+------------------------------+



In [183]:
customer_products_unique = (
    final_orders
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        collect_set("product_name").alias("unique_products")
    )
)

customer_products_unique.show(truncate=False)

+-----------+-------------+------------------------------+
|customer_id|customer_name|unique_products               |
+-----------+-------------+------------------------------+
|107        |Karan        |[Monitor]                     |
|103        |Sneha        |[Mouse, Monitor]              |
|104        |Priya        |[Laptop]                      |
|106        |Riya         |[Keyboard]                    |
|105        |Arjun        |[Mouse]                       |
|108        |Neha         |[Laptop]                      |
|101        |Rahul        |[Headphones, Keyboard, Laptop]|
|102        |Amit         |[Mouse, Headphones]           |
+-----------+-------------+------------------------------+



In [184]:
customer_test = spark.createDataFrame(
    [
        (101, "Rahul"),
        (102, "Amit"),
        (999, "Unknown Customer")
    ],
    ["customer_id", "name"]
)

In [185]:
#Inner Join
order_test = spark.createDataFrame(
    [
        (1, 101, 500),
        (2, 102, 700),
        (3, 103, 900)
    ],
    ["order_id", "customer_id", "amount"]
)

In [186]:
#Left Join
left = order_test.join(
    customer_test,
    "customer_id",
    "left"
)

left.show()

+-----------+--------+------+-----+
|customer_id|order_id|amount| name|
+-----------+--------+------+-----+
|        101|       1|   500|Rahul|
|        102|       2|   700| Amit|
|        103|       3|   900| NULL|
+-----------+--------+------+-----+



In [187]:
#Right Join
right = order_test.join(
    customer_test,
    "customer_id",
    "right"
)

right.show()

+-----------+--------+------+----------------+
|customer_id|order_id|amount|            name|
+-----------+--------+------+----------------+
|        101|       1|   500|           Rahul|
|        102|       2|   700|            Amit|
|        999|    NULL|  NULL|Unknown Customer|
+-----------+--------+------+----------------+



In [188]:
#Full Join
full = order_test.join(
    customer_test,
    "customer_id",
    "full"
)

full.show()

+-----------+--------+------+----------------+
|customer_id|order_id|amount|            name|
+-----------+--------+------+----------------+
|        101|       1|   500|           Rahul|
|        102|       2|   700|            Amit|
|        103|       3|   900|            NULL|
|        999|    NULL|  NULL|Unknown Customer|
+-----------+--------+------+----------------+



In [189]:
#Handle duplicate column names
orders.join(
    customers,
    orders.customer_id == customers.customer_id
)

DataFrame[order_id: bigint, customer_id: bigint, product_id: bigint, quantity: bigint, order_date: string, customer_id: bigint, customer_name: string, city: string]

In [190]:
o = orders.alias("o")
c = customers.alias("c")

joined = o.join(
    c,
    col("o.customer_id") == col("c.customer_id"),
    "left"
)

In [191]:
joined.select(
    col("o.order_id"),
    col("o.customer_id"),
    col("c.customer_name"),
    col("c.city")
).show()

+--------+-----------+-------------+---------+
|order_id|customer_id|customer_name|     city|
+--------+-----------+-------------+---------+
|    1001|        101|        Rahul|  Kolkata|
|    1003|        101|        Rahul|  Kolkata|
|    1002|        102|         Amit|    Delhi|
|    1004|        103|        Sneha|   Mumbai|
|    1005|        104|        Priya|     Pune|
|    1006|        105|        Arjun|Bangalore|
|    1007|        106|         Riya|  Chennai|
|    1009|        107|        Karan|Hyderabad|
|    1008|        102|         Amit|    Delhi|
|    1010|        108|         Neha|  Kolkata|
|    1012|        103|        Sneha|   Mumbai|
|    1011|        101|        Rahul|  Kolkata|
+--------+-----------+-------------+---------+



Most important thing today
Don't memorize syntax.
You should understand this flow:
                 RAW DATA
                    │
                    ▼
              select/filter
                    │
                    ▼
              withColumn()
                    │
                    ▼
                  JOIN
              ┌─────┴─────┐
              ▼           ▼
          customers    products
              │           │
              └─────┬─────┘
                    ▼
                groupBy()
                    │
                    ▼
                  agg()
                    │
                    ▼
                when()
                    │
                    ▼
              BUSINESS DATA


## 🪟 Day 06: Analytical Window Functions Deep Dive
### 🎯 Learning Objectives:
1. Construct PySpark **Window Specifications** combining `partitionBy()` and `orderBy()`.
2. Master ranking functions:
   - `row_number()`: Strict sequential numbering (deduplication & first/last order identification).
   - `rank()` vs `dense_rank()`: Handling ties with vs without gaps.
3. Compute time-series inter-row metrics using `lag()` (previous order value) and `lead()` (subsequent order).
4. Calculate financial customer deltas (spend growth/churn per transaction).
5. Build cumulative financial metrics (running revenue total via `unboundedPreceding` to `currentRow`).
6. Calculate rolling moving averages using sliding frame windows (`rowsBetween(-2, 0)`).
7. Solve Top-N analytical problems (Top 2 orders per customer, Top revenue customers per city).
---


PySpark Window Functions
Today's goal:
Orders
   │
   ▼
Window Specification
   │
   ├── row_number()
   ├── rank()
   ├── dense_rank()
   ├── lag()
   ├── lead()
   ├── running total
   └── first/last value
   │
   ▼
Advanced Analytics

By the end, you'll be able to solve things like:
- Latest record per customer
- First order of each customer
- Previous order
- Next order
- Difference between current and previous order
- Customer running revenue
- Top 3 products per category
- Ranking customers within a city


### ⚙️ Step 1: Initialize SparkSession for Day 06: Analytical Window Functions Deep Dive


In [194]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Day6_WindowFunctions")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [195]:
orders_data = [
    (1001, 101, "Rahul", "Kolkata", "Laptop", 1, 55000, "2026-09-01"),
    (1002, 102, "Amit", "Delhi", "Mouse", 2, 800, "2026-09-01"),
    (1003, 101, "Rahul", "Kolkata", "Keyboard", 1, 1500, "2026-09-03"),
    (1004, 103, "Sneha", "Mumbai", "Monitor", 2, 12000, "2026-09-03"),
    (1005, 101, "Rahul", "Kolkata", "Headphones", 2, 2500, "2026-09-05"),
    (1006, 102, "Amit", "Delhi", "Laptop", 1, 55000, "2026-09-06"),
    (1007, 104, "Priya", "Pune", "Laptop", 1, 55000, "2026-09-07"),
    (1008, 103, "Sneha", "Mumbai", "Mouse", 3, 800, "2026-09-08"),
    (1009, 101, "Rahul", "Kolkata", "Monitor", 1, 12000, "2026-09-09"),
    (1010, 102, "Amit", "Delhi", "Keyboard", 2, 1500, "2026-09-10"),
    (1011, 104, "Priya", "Pune", "Mouse", 2, 800, "2026-09-11"),
    (1012, 103, "Sneha", "Mumbai", "Laptop", 1, 55000, "2026-09-12"),
    (1013, 101, "Rahul", "Kolkata", "Laptop", 1, 55000, "2026-09-12"),
    (1014, 102, "Amit", "Delhi", "Monitor", 1, 12000, "2026-09-13"),
    (1015, 104, "Priya", "Pune", "Keyboard", 1, 1500, "2026-09-14")
]

columns = [
    "order_id",
    "customer_id",
    "customer_name",
    "city",
    "product",
    "quantity",
    "price",
    "order_date"
]

orders = spark.createDataFrame(
    orders_data,
    columns
)

In [196]:
orders = orders.withColumn(
    "order_date",
    to_date("order_date")
)

orders = orders.withColumn(
    "revenue",
    col("quantity") * col("price")
)

orders.show()

+--------+-----------+-------------+-------+----------+--------+-----+----------+-------+
|order_id|customer_id|customer_name|   city|   product|quantity|price|order_date|revenue|
+--------+-----------+-------------+-------+----------+--------+-----+----------+-------+
|    1001|        101|        Rahul|Kolkata|    Laptop|       1|55000|2026-09-01|  55000|
|    1002|        102|         Amit|  Delhi|     Mouse|       2|  800|2026-09-01|   1600|
|    1003|        101|        Rahul|Kolkata|  Keyboard|       1| 1500|2026-09-03|   1500|
|    1004|        103|        Sneha| Mumbai|   Monitor|       2|12000|2026-09-03|  24000|
|    1005|        101|        Rahul|Kolkata|Headphones|       2| 2500|2026-09-05|   5000|
|    1006|        102|         Amit|  Delhi|    Laptop|       1|55000|2026-09-06|  55000|
|    1007|        104|        Priya|   Pune|    Laptop|       1|55000|2026-09-07|  55000|
|    1008|        103|        Sneha| Mumbai|     Mouse|       3|  800|2026-09-08|   2400|
|    1009|

In [197]:
customer_window = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")

Window
│
├── partitionBy()
│       ↓
│   Divide data into groups
│
└── orderBy()
        ↓
    Order rows inside each group


In [199]:
orders_with_row_number = orders.withColumn(
    "row_number",
    row_number().over(customer_window)
)

orders_with_row_number.select(
    "customer_id",
    "order_id",
    "order_date",
    "row_number"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+--------+----------+----------+
|customer_id|order_id|order_date|row_number|
+-----------+--------+----------+----------+
|        101|    1001|2026-09-01|         1|
|        101|    1003|2026-09-03|         2|
|        101|    1005|2026-09-05|         3|
|        101|    1009|2026-09-09|         4|
|        101|    1013|2026-09-12|         5|
|        102|    1002|2026-09-01|         1|
|        102|    1006|2026-09-06|         2|
|        102|    1010|2026-09-10|         3|
|        102|    1014|2026-09-13|         4|
|        103|    1004|2026-09-03|         1|
|        103|    1008|2026-09-08|         2|
|        103|    1012|2026-09-12|         3|
|        104|    1007|2026-09-07|         1|
|        104|    1011|2026-09-11|         2|
|        104|    1015|2026-09-14|         3|
+-----------+--------+----------+----------+



In [200]:
first_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(customer_window)
    )
    .filter(
        col("rn") == 1
    )
)

first_orders.select(
    "customer_id",
    "customer_name",
    "order_id",
    "order_date",
    "revenue"
).show()

+-----------+-------------+--------+----------+-------+
|customer_id|customer_name|order_id|order_date|revenue|
+-----------+-------------+--------+----------+-------+
|        101|        Rahul|    1001|2026-09-01|  55000|
|        102|         Amit|    1002|2026-09-01|   1600|
|        103|        Sneha|    1004|2026-09-03|  24000|
|        104|        Priya|    1007|2026-09-07|  55000|
+-----------+-------------+--------+----------+-------+



In [201]:
latest_window = Window \
    .partitionBy("customer_id") \
    .orderBy(col("order_date").desc())

In [202]:
latest_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(latest_window)
    )
    .filter(
        col("rn") == 1
    )
)

In [203]:
latest_orders.select(
    "customer_id",
    "customer_name",
    "order_id",
    "order_date",
    "revenue"
).show()

+-----------+-------------+--------+----------+-------+
|customer_id|customer_name|order_id|order_date|revenue|
+-----------+-------------+--------+----------+-------+
|        101|        Rahul|    1013|2026-09-12|  55000|
|        102|         Amit|    1014|2026-09-13|  12000|
|        103|        Sneha|    1012|2026-09-12|  55000|
|        104|        Priya|    1015|2026-09-14|   1500|
+-----------+-------------+--------+----------+-------+



In [204]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

In [205]:
revenue_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("revenue").desc())
)

ranked_orders = orders.withColumn(
    "revenue_rank",
    rank().over(revenue_window)
)

In [206]:
ranked_orders.select(
    "customer_id",
    "order_id",
    "revenue",
    "revenue_rank"
).orderBy(
    "customer_id",
    "revenue_rank"
).show()

+-----------+--------+-------+------------+
|customer_id|order_id|revenue|revenue_rank|
+-----------+--------+-------+------------+
|        101|    1001|  55000|           1|
|        101|    1013|  55000|           1|
|        101|    1009|  12000|           3|
|        101|    1005|   5000|           4|
|        101|    1003|   1500|           5|
|        102|    1006|  55000|           1|
|        102|    1014|  12000|           2|
|        102|    1010|   3000|           3|
|        102|    1002|   1600|           4|
|        103|    1012|  55000|           1|
|        103|    1004|  24000|           2|
|        103|    1008|   2400|           3|
|        104|    1007|  55000|           1|
|        104|    1011|   1600|           2|
|        104|    1015|   1500|           3|
+-----------+--------+-------+------------+



RANK vs DENSE_RANK
This is important.
Suppose revenues are:
55000
55000
12000
800

rank()
55000 → 1
55000 → 1
12000 → 3
800   → 4

dense_rank()
55000 → 1
55000 → 1
12000 → 2
800   → 3


In [208]:
rank_test = orders.withColumn(
    "rank",
    rank().over(revenue_window)
).withColumn(
    "dense_rank",
    dense_rank().over(revenue_window)
)

rank_test.select(
    "customer_id",
    "order_id",
    "revenue",
    "rank",
    "dense_rank"
).orderBy(
    "customer_id",
    "rank"
).show()

+-----------+--------+-------+----+----------+
|customer_id|order_id|revenue|rank|dense_rank|
+-----------+--------+-------+----+----------+
|        101|    1001|  55000|   1|         1|
|        101|    1013|  55000|   1|         1|
|        101|    1009|  12000|   3|         2|
|        101|    1005|   5000|   4|         3|
|        101|    1003|   1500|   5|         4|
|        102|    1006|  55000|   1|         1|
|        102|    1014|  12000|   2|         2|
|        102|    1010|   3000|   3|         3|
|        102|    1002|   1600|   4|         4|
|        103|    1012|  55000|   1|         1|
|        103|    1004|  24000|   2|         2|
|        103|    1008|   2400|   3|         3|
|        104|    1007|  55000|   1|         1|
|        104|    1011|   1600|   2|         2|
|        104|    1015|   1500|   3|         3|
+-----------+--------+-------+----+----------+



In [209]:
rank_test.select(
    "customer_id",
    "order_id",
    "revenue",
    "rank",
    "dense_rank"
).orderBy(
    "customer_id",
    col("revenue").desc()
).show()

+-----------+--------+-------+----+----------+
|customer_id|order_id|revenue|rank|dense_rank|
+-----------+--------+-------+----+----------+
|        101|    1001|  55000|   1|         1|
|        101|    1013|  55000|   1|         1|
|        101|    1009|  12000|   3|         2|
|        101|    1005|   5000|   4|         3|
|        101|    1003|   1500|   5|         4|
|        102|    1006|  55000|   1|         1|
|        102|    1014|  12000|   2|         2|
|        102|    1010|   3000|   3|         3|
|        102|    1002|   1600|   4|         4|
|        103|    1012|  55000|   1|         1|
|        103|    1004|  24000|   2|         2|
|        103|    1008|   2400|   3|         3|
|        104|    1007|  55000|   1|         1|
|        104|    1011|   1600|   2|         2|
|        104|    1015|   1500|   3|         3|
+-----------+--------+-------+----+----------+



In [210]:
# Now we get to the part you already know from SQL.
# lag() gives the previous row.
customer_date_window = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")

In [211]:
orders_lag = orders.withColumn(
    "previous_revenue",
    lag("revenue", 1).over(customer_date_window)
)

In [212]:
orders_lag.select(
    "customer_id",
    "order_id",
    "order_date",
    "revenue",
    "previous_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+--------+----------+-------+----------------+
|customer_id|order_id|order_date|revenue|previous_revenue|
+-----------+--------+----------+-------+----------------+
|        101|    1001|2026-09-01|  55000|            NULL|
|        101|    1003|2026-09-03|   1500|           55000|
|        101|    1005|2026-09-05|   5000|            1500|
|        101|    1009|2026-09-09|  12000|            5000|
|        101|    1013|2026-09-12|  55000|           12000|
|        102|    1002|2026-09-01|   1600|            NULL|
|        102|    1006|2026-09-06|  55000|            1600|
|        102|    1010|2026-09-10|   3000|           55000|
|        102|    1014|2026-09-13|  12000|            3000|
|        103|    1004|2026-09-03|  24000|            NULL|
|        103|    1008|2026-09-08|   2400|           24000|
|        103|    1012|2026-09-12|  55000|            2400|
|        104|    1007|2026-09-07|  55000|            NULL|
|        104|    1011|2026-09-11|   1600|           5500

In [213]:
# Revenue difference
# Now calculate:
# current revenue - previous revenue
orders_lag = orders_lag.withColumn(
    "revenue_difference",
    col("revenue") - col("previous_revenue")
)

orders_lag.select(
    "customer_id",
    "order_date",
    "revenue",
    "previous_revenue",
    "revenue_difference"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+----------------+------------------+
|customer_id|order_date|revenue|previous_revenue|revenue_difference|
+-----------+----------+-------+----------------+------------------+
|        101|2026-09-01|  55000|            NULL|              NULL|
|        101|2026-09-03|   1500|           55000|            -53500|
|        101|2026-09-05|   5000|            1500|              3500|
|        101|2026-09-09|  12000|            5000|              7000|
|        101|2026-09-12|  55000|           12000|             43000|
|        102|2026-09-01|   1600|            NULL|              NULL|
|        102|2026-09-06|  55000|            1600|             53400|
|        102|2026-09-10|   3000|           55000|            -52000|
|        102|2026-09-13|  12000|            3000|              9000|
|        103|2026-09-03|  24000|            NULL|              NULL|
|        103|2026-09-08|   2400|           24000|            -21600|
|        103|2026-09-12|  55000|  

In [214]:
# lead() gives the next row.
orders_lead = orders.withColumn(
    "next_revenue",
    lead("revenue", 1).over(customer_date_window)
)

orders_lead.select(
    "customer_id",
    "order_date",
    "revenue",
    "next_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+------------+
|customer_id|order_date|revenue|next_revenue|
+-----------+----------+-------+------------+
|        101|2026-09-01|  55000|        1500|
|        101|2026-09-03|   1500|        5000|
|        101|2026-09-05|   5000|       12000|
|        101|2026-09-09|  12000|       55000|
|        101|2026-09-12|  55000|        NULL|
|        102|2026-09-01|   1600|       55000|
|        102|2026-09-06|  55000|        3000|
|        102|2026-09-10|   3000|       12000|
|        102|2026-09-13|  12000|        NULL|
|        103|2026-09-03|  24000|        2400|
|        103|2026-09-08|   2400|       55000|
|        103|2026-09-12|  55000|        NULL|
|        104|2026-09-07|  55000|        1600|
|        104|2026-09-11|   1600|        1500|
|        104|2026-09-14|   1500|        NULL|
+-----------+----------+-------+------------+



Conceptually:
LAG

current
   ↑
previous


LEAD

current
   ↓
next


Running total 🔥🔥
Now calculate each customer's cumulative revenue.
This is extremely useful.
We need a window with:
PARTITION BY customer
ORDER BY date
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW


In [217]:
running_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

In [218]:
orders_running = orders.withColumn(
    "running_revenue",
    sum("revenue").over(running_window)
)

In [219]:
orders_running.select(
    "customer_id",
    "order_date",
    "revenue",
    "running_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+---------------+
|customer_id|order_date|revenue|running_revenue|
+-----------+----------+-------+---------------+
|        101|2026-09-01|  55000|          55000|
|        101|2026-09-03|   1500|          56500|
|        101|2026-09-05|   5000|          61500|
|        101|2026-09-09|  12000|          73500|
|        101|2026-09-12|  55000|         128500|
|        102|2026-09-01|   1600|           1600|
|        102|2026-09-06|  55000|          56600|
|        102|2026-09-10|   3000|          59600|
|        102|2026-09-13|  12000|          71600|
|        103|2026-09-03|  24000|          24000|
|        103|2026-09-08|   2400|          26400|
|        103|2026-09-12|  55000|          81400|
|        104|2026-09-07|  55000|          55000|
|        104|2026-09-11|   1600|          56600|
|        104|2026-09-14|   1500|          58100|
+-----------+----------+-------+---------------+



In [220]:
orders_running = orders_running.withColumn(
    "running_quantity",
    sum("quantity").over(running_window)
)

orders_running.select(
    "customer_id",
    "order_date",
    "quantity",
    "running_quantity"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+--------+----------------+
|customer_id|order_date|quantity|running_quantity|
+-----------+----------+--------+----------------+
|        101|2026-09-01|       1|               1|
|        101|2026-09-03|       1|               2|
|        101|2026-09-05|       2|               4|
|        101|2026-09-09|       1|               5|
|        101|2026-09-12|       1|               6|
|        102|2026-09-01|       2|               2|
|        102|2026-09-06|       1|               3|
|        102|2026-09-10|       2|               5|
|        102|2026-09-13|       1|               6|
|        103|2026-09-03|       2|               2|
|        103|2026-09-08|       3|               5|
|        103|2026-09-12|       1|               6|
|        104|2026-09-07|       1|               1|
|        104|2026-09-11|       2|               3|
|        104|2026-09-14|       1|               4|
+-----------+----------+--------+----------------+



In [221]:
moving_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(-2, 0)
)

In [222]:
orders_moving = orders.withColumn(
    "moving_avg_revenue",
    avg("revenue").over(moving_window)
)

In [223]:
orders_moving.select(
    "customer_id",
    "order_date",
    "revenue",
    "moving_avg_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+------------------+
|customer_id|order_date|revenue|moving_avg_revenue|
+-----------+----------+-------+------------------+
|        101|2026-09-01|  55000|           55000.0|
|        101|2026-09-03|   1500|           28250.0|
|        101|2026-09-05|   5000|           20500.0|
|        101|2026-09-09|  12000| 6166.666666666667|
|        101|2026-09-12|  55000|           24000.0|
|        102|2026-09-01|   1600|            1600.0|
|        102|2026-09-06|  55000|           28300.0|
|        102|2026-09-10|   3000|19866.666666666668|
|        102|2026-09-13|  12000|23333.333333333332|
|        103|2026-09-03|  24000|           24000.0|
|        103|2026-09-08|   2400|           13200.0|
|        103|2026-09-12|  55000|27133.333333333332|
|        104|2026-09-07|  55000|           55000.0|
|        104|2026-09-11|   1600|           28300.0|
|        104|2026-09-14|   1500|19366.666666666668|
+-----------+----------+-------+------------------+



Current row
+
previous 2 rows


In [225]:
top_orders_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("revenue").desc())
)

In [226]:
top_2_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(top_orders_window)
    )
    .filter(
        col("rn") <= 2
    )
)

In [227]:
top_2_orders.select(
    "customer_id",
    "order_id",
    "revenue",
    "rn"
).orderBy(
    "customer_id",
    "rn"
).show()

+-----------+--------+-------+---+
|customer_id|order_id|revenue| rn|
+-----------+--------+-------+---+
|        101|    1001|  55000|  1|
|        101|    1013|  55000|  2|
|        102|    1006|  55000|  1|
|        102|    1014|  12000|  2|
|        103|    1012|  55000|  1|
|        103|    1004|  24000|  2|
|        104|    1007|  55000|  1|
|        104|    1011|   1600|  2|
+-----------+--------+-------+---+



In [228]:
# Rank customers within their city
# Now we'll combine aggregation + window.
# First calculate customer revenue.
customer_summary = (
    orders
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)

customer_summary.show()

+-----------+-------------+-------+-------------+------------+
|customer_id|customer_name|   city|total_revenue|total_orders|
+-----------+-------------+-------+-------------+------------+
|        102|         Amit|  Delhi|        71600|           4|
|        104|        Priya|   Pune|        58100|           3|
|        103|        Sneha| Mumbai|        81400|           3|
|        101|        Rahul|Kolkata|       128500|           5|
+-----------+-------------+-------+-------------+------------+



In [229]:
city_window = (
    Window
    .partitionBy("city")
    .orderBy(col("total_revenue").desc())
)

In [230]:
customer_ranked = customer_summary.withColumn(
    "city_rank",
    dense_rank().over(city_window)
)

In [231]:
customer_ranked.orderBy(
    "city",
    "city_rank"
).show()

+-----------+-------------+-------+-------------+------------+---------+
|customer_id|customer_name|   city|total_revenue|total_orders|city_rank|
+-----------+-------------+-------+-------------+------------+---------+
|        102|         Amit|  Delhi|        71600|           4|        1|
|        101|        Rahul|Kolkata|       128500|           5|        1|
|        103|        Sneha| Mumbai|        81400|           3|        1|
|        104|        Priya|   Pune|        58100|           3|        1|
+-----------+-------------+-------+-------------+------------+---------+



In [232]:
top_city_customers = customer_ranked.filter(
    col("city_rank") == 1
)

top_city_customers.show()

+-----------+-------------+-------+-------------+------------+---------+
|customer_id|customer_name|   city|total_revenue|total_orders|city_rank|
+-----------+-------------+-------+-------------+------------+---------+
|        102|         Amit|  Delhi|        71600|           4|        1|
|        101|        Rahul|Kolkata|       128500|           5|        1|
|        103|        Sneha| Mumbai|        81400|           3|        1|
|        104|        Priya|   Pune|        58100|           3|        1|
+-----------+-------------+-------+-------------+------------+---------+



GROUP BY
    ↓
calculate metric
    ↓
WINDOW
    ↓
RANK
    ↓
filter rank = 1


## 🏗️ Day 07: Modular Production Pipeline Architecture & Reusable ETL Functions
### 🎯 Learning Objectives:
1. Refactor monolithic script code into modular, testable, and reusable Python functions.
2. Build dedicated single-responsibility components:
   - `ingest_csv()`: Standardized ingestion with schema enforcement.
   - `add_metadata()`: Lineage and ingestion timestamps.
   - `validate_orders()`: Contract validation returning valid and rejected tuples.
   - `deduplicate_orders()`: Idempotent deduplication.
   - `transform_orders()`: Business feature engineering.
   - `create_product_report()`: Gold analytical mart generation.
   - `write_parquet()`: Reusable partitioned persistence layer.
3. Create a master pipeline driver `run_pipeline(spark, source_path)` that orchestrates the workflow.
4. Output structured data quality metrics and verify contract compliance.
---


### ⚙️ Step 1: Initialize SparkSession for Day 07: Modular Production Pipeline Architecture & Reusable ETL Functions


In [234]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

import pandas as pd
from pathlib import Path
import logging
import os

# Ensure HADOOP_HOME is set for Windows
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] += os.pathsep + r'C:\hadoop\bin'

spark = (
    SparkSession.builder
    .appName("Day7_Reusable_ETL")
    .master("local[*]")
    .getOrCreate()
)

BASE = Path("day7_project")

for folder in ["source", "bronze", "silver", "gold", "rejected"]:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

print("Environment ready")


Environment ready


In [235]:
data = {
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1008, 1009
    ],
    "customer_id": [
        101, 102, 103, 104, 105,
        101, None, 106, 106, 107
    ],
    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor",
        "Laptop", "Mouse", "Keyboard",
        "Monitor", "Monitor", "Laptop"
    ],
    "quantity": [1, 2, 1, 2, -1, 3, 1, 2, 2, 1],
    "price": [
        55000, 800, 1500, 12000, 55000,
        800, 1500, 12000, 12000, 55000
    ],
    "order_date": [
        "2026-10-01", "2026-10-01", "2026-10-02",
        "2026-10-02", "2026-10-03", "2026-10-03",
        "2026-10-03", "2026-10-04", "2026-10-04",
        "INVALID"
    ]
}

df_orders = pd.DataFrame(data)
df_orders["customer_id"] = df_orders["customer_id"].astype("Int64")

df_orders.to_csv(
    BASE / "source" / "orders.csv",
    index=False
)

print("Practice CSV created")


Practice CSV created


In [236]:
ORDER_SCHEMA = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [237]:
def ingest_csv(spark, file_path, schema):
    df = (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(str(file_path))
    )

    return df

In [238]:
raw_df = ingest_csv(
    spark,
    BASE / "source" / "orders.csv",
    ORDER_SCHEMA
)

raw_df.show()

+--------+-----------+--------+--------+-------+----------+
|order_id|customer_id| product|quantity|  price|order_date|
+--------+-----------+--------+--------+-------+----------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|
|    1005|        105|  Laptop|      -1|55000.0|2026-10-03|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|
|    1007|       NULL|Keyboard|       1| 1500.0|2026-10-03|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|
|    1009|        107|  Laptop|       1|55000.0|   INVALID|
+--------+-----------+--------+--------+-------+----------+



In [239]:
def add_metadata(df):
    return (
        df
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_file",
            F.input_file_name()
        )
    )

In [240]:
bronze_df = add_metadata(raw_df)

bronze_df.show(truncate=False)

+--------+-----------+--------+--------+-------+----------+--------------------------+---------------------------------------------------------------------------------------+
|order_id|customer_id|product |quantity|price  |order_date|ingestion_timestamp       |source_file                                                                            |
+--------+-----------+--------+--------+-------+----------+--------------------------+---------------------------------------------------------------------------------------+
|1001    |101        |Laptop  |1       |55000.0|2026-10-01|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%20Learning/Pyspark/day7_project/source/orders.csv|
|1002    |102        |Mouse   |2       |800.0  |2026-10-01|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%20Learning/Pyspark/day7_project/source/orders.csv|
|1003    |103        |Keyboard|1       |1500.0 |2026-10-02|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%2

In [241]:
def validate_orders(df):

    checked = (
        df
        .withColumn(
            "parsed_date",
            F.try_to_timestamp(
                F.col("order_date"),
                F.lit("yyyy-MM-dd")
            ).cast("date")
        )
        .withColumn(
            "validation_errors",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("order_id").isNull(),
                        F.lit("Missing order ID")
                    ),
                    F.when(
                        F.col("customer_id").isNull(),
                        F.lit("Missing customer ID")
                    ),
                    F.when(
                        F.col("quantity").isNull() |
                        (F.col("quantity") <= 0),
                        F.lit("Invalid quantity")
                    ),
                    F.when(
                        F.col("price").isNull() |
                        (F.col("price") <= 0),
                        F.lit("Invalid price")
                    ),
                    F.when(
                        F.col("parsed_date").isNull(),
                        F.lit("Invalid date")
                    )
                )
            )
        )
    )

    valid = checked.filter(
        F.size("validation_errors") == 0
    )

    rejected = checked.filter(
        F.size("validation_errors") > 0
    )

    return valid, rejected

In [242]:
valid_df, rejected_df = validate_orders(bronze_df)

valid_df.show()
rejected_df.select(
    "order_id",
    "validation_errors"
).show(truncate=False)

+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-----------+-----------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|         source_file|parsed_date|validation_errors|
+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-----------+-----------------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-01|               []|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-01|               []|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-02|               []|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-02|               []|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|2026-10-08 07:43:

In [243]:
def deduplicate_orders(df):

    window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("ingestion_timestamp").desc(),
            F.col("source_file").asc()
        )
    )

    ranked = df.withColumn(
        "rn",
        F.row_number().over(window)
    )

    unique = ranked.filter(F.col("rn") == 1).drop("rn")

    duplicates = ranked.filter(F.col("rn") > 1).drop("rn")

    return unique, duplicates

In [244]:
unique_df, duplicates_df = deduplicate_orders(valid_df)

print("Unique:", unique_df.count())
print("Duplicates:", duplicates_df.count())

Unique: 6
Duplicates: 1


In [245]:
def transform_orders(df):
    return (
        df
        .withColumn(
            "product",
            F.initcap(F.trim(F.col("product")))
        )
        .withColumn(
            "order_date",
            F.col("parsed_date")
        )
        .withColumn(
            "revenue",
            F.col("quantity") * F.col("price")
        )
        .drop("parsed_date", "validation_errors")
    )

In [246]:
silver_df = transform_orders(unique_df)

silver_df.select(
    "order_id",
    "product",
    "quantity",
    "price",
    "revenue"
).show()

+--------+--------+--------+-------+-------+
|order_id| product|quantity|  price|revenue|
+--------+--------+--------+-------+-------+
|    1001|  Laptop|       1|55000.0|55000.0|
|    1002|   Mouse|       2|  800.0| 1600.0|
|    1003|Keyboard|       1| 1500.0| 1500.0|
|    1004| Monitor|       2|12000.0|24000.0|
|    1006|   Mouse|       3|  800.0| 2400.0|
|    1008| Monitor|       2|12000.0|24000.0|
+--------+--------+--------+-------+-------+



In [247]:
def write_parquet(df, path, partitions=None):

    writer = df.write.mode("overwrite")

    if partitions:
        writer = writer.partitionBy(*partitions)

    writer.parquet(str(path))

In [248]:
def create_product_report(df):
    return (
        df
        .groupBy("product")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("revenue").alias("total_revenue"),
            F.avg("revenue").alias("avg_order_value")
        )
        .orderBy(F.desc("total_revenue"))
    )

In [249]:
gold_df = create_product_report(silver_df)

gold_df.show()

+--------+------------+--------------+-------------+---------------+
| product|total_orders|total_quantity|total_revenue|avg_order_value|
+--------+------------+--------------+-------------+---------------+
|  Laptop|           1|             1|      55000.0|        55000.0|
| Monitor|           2|             4|      48000.0|        24000.0|
|   Mouse|           2|             5|       4000.0|         2000.0|
|Keyboard|           1|             1|       1500.0|         1500.0|
+--------+------------+--------------+-------------+---------------+



### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [250]:
def run_pipeline(spark, source_path):

    # 1. Ingestion
    raw = ingest_csv(
        spark,
        source_path,
        ORDER_SCHEMA
    )

    # 2. Metadata
    bronze = add_metadata(raw)

    # 3. Validation
    valid, rejected = validate_orders(bronze)

    # 4. Deduplication
    unique, duplicates = deduplicate_orders(valid)

    # 5. Transformation
    silver = transform_orders(unique)

    # 6. Aggregation
    gold = create_product_report(silver)

    # 7. Calculate metrics
    total = bronze.count()
    invalid_count = rejected.count()
    duplicate_count = duplicates.count()
    accepted = silver.count()

    assert total == (
        invalid_count + duplicate_count + accepted
    ), "Record reconciliation failed"

    # 8. Save outputs
    write_parquet(bronze, BASE / "bronze")
    write_parquet(rejected, BASE / "rejected" / "invalid")
    write_parquet(duplicates, BASE / "rejected" / "duplicates")
    write_parquet(silver, BASE / "silver")
    write_parquet(gold, BASE / "gold")

    return {
        "total": total,
        "accepted": accepted,
        "invalid": invalid_count,
        "duplicates": duplicate_count
    }

In [251]:
report = run_pipeline(
    spark,
    BASE / "source" / "orders.csv"
)

print("DATA QUALITY REPORT")

for key, value in report.items():
    print(f"{key}: {value}")

DATA QUALITY REPORT
total: 10
accepted: 6
invalid: 3
duplicates: 1


In [252]:
saved_silver = spark.read.parquet(
    str(BASE / "silver")
)

saved_gold = spark.read.parquet(
    str(BASE / "gold")
)

saved_silver.show()
saved_gold.show()

+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|         source_file|revenue|
+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...|55000.0|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 1600.0|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 1500.0|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...|24000.0|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|2026-10-08 07:43:...|file:///C:/Users/...| 2400.0|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|2026-10-08 07:43:...|file:///C:/Users/...|24000.0|
+--------+

## ⚙️ Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing
### 🎯 Learning Objectives:
1. Decouple pipeline parameters into an external centralized `CONFIG` management structure.
2. Implement enterprise logging with dual handlers (console streaming + persistent `pipeline.log` file).
3. Implement fail-safe exception handling with structured error logging.
4. Enforce strict **Data Quality SLAs** (e.g. automatically aborting pipeline if rejection rate exceeds 30%).
5. Generate an auditable execution report saved as `pipeline_report.json`.
6. Construct an automated **Unit Test Suite** with PySpark assertions:
   - Test 1: Record Reconciliation (`Total = Valid + Rejected`)
   - Test 2: Schema Cleanliness (Zero invalid customer IDs in Silver)
   - Test 3: Business Boundary Validation (Zero negative revenues)
   - Test 4: Financial Calculation Integrity (`Revenue == Quantity * Price`).
---


Logging + Configuration + Error Handling + Testing
The goal is:
                    Pipeline
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
          Config     Logging   Validation
             │         │         │
             └─────────┼─────────┘
                       ▼
                 ETL Processing
                       │
              ┌────────┴────────┐
              ▼                 ▼
           SUCCESS             ERROR
              │                 │
              ▼                 ▼
          Output Data       Error Log

Today you'll learn four things:
1. Configuration — don't hardcode paths everywhere.
2. Logging — know exactly what your pipeline did.
3. Exception handling — fail gracefully.
4. Testing — verify that your pipeline produces correct results.


### ⚙️ Step 1: Initialize SparkSession for Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing


In [254]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

from pathlib import Path
import logging
import json
from datetime import datetime

spark = (
    SparkSession.builder
    .appName("Day8_PipelineEngineering")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [255]:
CONFIG = {
    "project_name": "ecommerce_pipeline",

    "source_path": "day7_project/source/orders.csv",

    "bronze_path": "day7_project/bronze",

    "silver_path": "day7_project/silver",

    "gold_path": "day7_project/gold",

    "rejected_path": "day7_project/rejected",

    "log_path": "day7_project/logs"
}

CONFIG

{'project_name': 'ecommerce_pipeline',
 'source_path': 'day7_project/source/orders.csv',
 'bronze_path': 'day7_project/bronze',
 'silver_path': 'day7_project/silver',
 'gold_path': 'day7_project/gold',
 'rejected_path': 'day7_project/rejected',
 'log_path': 'day7_project/logs'}

In [256]:
for key in [
    "bronze_path",
    "silver_path",
    "gold_path",
    "rejected_path",
    "log_path"
]:
    Path(CONFIG[key]).mkdir(
        parents=True,
        exist_ok=True
    )

print("Directories ready")

Directories ready


In [257]:
from pathlib import Path
import logging

# 1. Ensure directory exists
log_dir = Path(CONFIG["log_path"])
log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / "pipeline.log"

# 2. Configure logger and clear existing handlers if re-running in a notebook
logger = logging.getLogger("EcommercePipeline")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

# 3. Add File Handler
file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

logger.info("Pipeline logger initialized")
print("Log file location:", log_file.resolve())

2026-10-08 07:43:35,761 - INFO - Pipeline logger initialized


Log file location: C:\Users\Me\Documents\Machine Learning\Pyspark\day7_project\logs\pipeline.log


In [258]:
logger.info("Starting Day 8 pipeline")
logger.warning("This is a test warning")
logger.error("This is a test error")

2026-10-08 07:43:35,790 - INFO - Starting Day 8 pipeline
2026-10-08 07:43:35,793 - WARNING - This is a test warning
2026-10-08 07:43:35,796 - ERROR - This is a test error


In [259]:
try:
    df = spark.read.csv(
        "does_not_exist.csv"
    )

    print(df.count())

except Exception as e:
    logger.error(
        f"Failed to read source file: {e}"
    )

    print("Pipeline failed while reading source")

2026-10-08 07:43:35,878 - ERROR - Failed to read source file: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/Me/Documents/Machine Learning/Pyspark/does_not_exist.csv. SQLSTATE: 42K03


Pipeline failed while reading source


In [260]:
def ingest_data(spark, config):

    source_path = config["source_path"]

    logger.info(
        f"Starting ingestion: {source_path}"
    )

    try:

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
        )

        record_count = df.count()

        logger.info(
            f"Ingestion successful. Records: {record_count}"
        )

        return df

    except Exception as e:

        logger.exception(
            f"Ingestion failed: {e}"
        )

        raise

catching the error
       ↓
logging it
       ↓
raising it again


In [262]:
raw_df = ingest_data(
    spark,
    CONFIG
)

raw_df.show()

2026-10-08 07:43:35,941 - INFO - Starting ingestion: day7_project/source/orders.csv
2026-10-08 07:43:36,291 - INFO - Ingestion successful. Records: 10


+--------+-----------+--------+--------+-----+----------+
|order_id|customer_id| product|quantity|price|order_date|
+--------+-----------+--------+--------+-----+----------+
|    1001|        101|  Laptop|       1|55000|2026-10-01|
|    1002|        102|   Mouse|       2|  800|2026-10-01|
|    1003|        103|Keyboard|       1| 1500|2026-10-02|
|    1004|        104| Monitor|       2|12000|2026-10-02|
|    1005|        105|  Laptop|      -1|55000|2026-10-03|
|    1006|        101|   Mouse|       3|  800|2026-10-03|
|    1007|       NULL|Keyboard|       1| 1500|2026-10-03|
|    1008|        106| Monitor|       2|12000|2026-10-04|
|    1008|        106| Monitor|       2|12000|2026-10-04|
|    1009|        107|  Laptop|       1|55000|   INVALID|
+--------+-----------+--------+--------+-----+----------+



In [263]:
from pyspark.sql import functions as F
import builtins

def validate_data(df):
    logger.info("Starting data validation")
    
    checked = (
        df
        .withColumn(
            "parsed_date", 
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd"))
        )
        .withColumn(
            "validation_errors",
            F.array_compact(
                F.array(
                    F.when(F.col("order_id").isNull(), F.lit("Missing order ID")),
                    F.when(F.col("customer_id").isNull(), F.lit("Missing customer ID")),
                    F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), F.lit("Invalid quantity")),
                    F.when(F.col("price").isNull() | (F.col("price") <= 0), F.lit("Invalid price")),
                    F.when(F.col("order_date").isNotNull() & F.col("parsed_date").isNull(), F.lit("Invalid date format"))
                )
            )
        )
    )
    
    # Separate valid rows (no errors) from rejected rows (has errors)
    valid_df = checked.filter(F.size("validation_errors") == 0)
    rejected_df = checked.filter(F.size("validation_errors") > 0)
    
    valid_count = valid_df.count()
    rejected_count = rejected_df.count()
    
    logger.info(
        f"Validation complete | "
        f"Valid: {valid_count} | "
        f"Rejected: {rejected_count}"
    )
    
    return valid_df, rejected_df


In [264]:
valid_df, rejected_df = validate_data(
    raw_df
)

valid_df.show()

2026-10-08 07:43:36,453 - INFO - Starting data validation
2026-10-08 07:43:36,841 - INFO - Validation complete | Valid: 7 | Rejected: 3


+--------+-----------+--------+--------+-----+----------+-------------------+-----------------+
|order_id|customer_id| product|quantity|price|order_date|        parsed_date|validation_errors|
+--------+-----------+--------+--------+-----+----------+-------------------+-----------------+
|    1001|        101|  Laptop|       1|55000|2026-10-01|2026-10-01 00:00:00|               []|
|    1002|        102|   Mouse|       2|  800|2026-10-01|2026-10-01 00:00:00|               []|
|    1003|        103|Keyboard|       1| 1500|2026-10-02|2026-10-02 00:00:00|               []|
|    1004|        104| Monitor|       2|12000|2026-10-02|2026-10-02 00:00:00|               []|
|    1006|        101|   Mouse|       3|  800|2026-10-03|2026-10-03 00:00:00|               []|
|    1008|        106| Monitor|       2|12000|2026-10-04|2026-10-04 00:00:00|               []|
|    1008|        106| Monitor|       2|12000|2026-10-04|2026-10-04 00:00:00|               []|
+--------+-----------+--------+--------+

In [265]:
rejected_df.select(
    "order_id",
    "validation_errors"
).show(truncate=False)

+--------+---------------------+
|order_id|validation_errors    |
+--------+---------------------+
|1005    |[Invalid quantity]   |
|1007    |[Missing customer ID]|
|1009    |[Invalid date format]|
+--------+---------------------+



In [266]:
total_records = raw_df.count()
valid_records = valid_df.count()
rejected_records = rejected_df.count()

rejection_rate = (
    rejected_records / total_records
    if total_records > 0
    else 0
)

logger.info(
    f"Pipeline metrics | "
    f"Total={total_records} | "
    f"Valid={valid_records} | "
    f"Rejected={rejected_records} | "
    f"RejectionRate={rejection_rate:.2%}"
)

print("Total:", total_records)
print("Valid:", valid_records)
print("Rejected:", rejected_records)
print("Rejection rate:", builtins.round(rejection_rate * 100, 2), "%")


2026-10-08 07:43:37,618 - INFO - Pipeline metrics | Total=10 | Valid=7 | Rejected=3 | RejectionRate=30.00%


Total: 10
Valid: 7
Rejected: 3
Rejection rate: 30.0 %


In [267]:
MAX_REJECTION_RATE = 0.30

if rejection_rate > MAX_REJECTION_RATE:

    logger.error(
        f"Rejection rate {rejection_rate:.2%} "
        f"exceeded allowed limit "
        f"{MAX_REJECTION_RATE:.2%}"
    )

    raise ValueError(
        "Data quality threshold exceeded"
    )

else:

    logger.info(
        "Data quality threshold passed"
    )

2026-10-08 07:43:37,638 - INFO - Data quality threshold passed


In [268]:
pipeline_report = {
    "pipeline": CONFIG["project_name"],
    "run_time": datetime.now().isoformat(),
    "records_received": total_records,
    "records_valid": valid_records,
    "records_rejected": rejected_records,
    "rejection_rate": builtins.round(
        rejection_rate,
        4
    ),
    "status": "SUCCESS"
}

pipeline_report


{'pipeline': 'ecommerce_pipeline',
 'run_time': '2026-10-08T07:43:37.660756',
 'records_received': 10,
 'records_valid': 7,
 'records_rejected': 3,
 'rejection_rate': 0.3,
 'status': 'SUCCESS'}

In [269]:
report_path = (
    Path(CONFIG["log_path"])
    / "pipeline_report.json"
)

with open(
    report_path,
    "w"
) as f:

    json.dump(
        pipeline_report,
        f,
        indent=4
    )

print(
    f"Report saved to {report_path}"
)

Report saved to day7_project\logs\pipeline_report.json


day7_project/
│
├── logs/
│   ├── pipeline.log
│   └── pipeline_report.json


In [271]:
assert total_records == (
    valid_records + rejected_records
)

print("✅ Record reconciliation passed")

✅ Record reconciliation passed


In [272]:
assert rejected_records > 0

print("✅ Invalid-record detection passed")

✅ Invalid-record detection passed


In [273]:
transformed_df = (
    valid_df
    .withColumn(
        "order_date",
        F.col("parsed_date")
    )
    .withColumn(
        "revenue",
        F.col("quantity") * F.col("price")
    )
)

In [274]:
negative_revenue = transformed_df.filter(
    F.col("revenue") < 0
).count()

assert negative_revenue == 0

print("✅ Revenue validation passed")

✅ Revenue validation passed


### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [275]:
def run_pipeline(spark, config):

    start_time = datetime.now()

    try:

        logger.info("========== PIPELINE START ==========")

        raw_df = ingest_data(spark, config)
        valid_df, rejected_df = validate_data(raw_df)

        total_records = raw_df.count()
        valid_records = valid_df.count()
        rejected_records = rejected_df.count()
        rejection_rate = rejected_records / total_records if total_records > 0 else 0.0

        assert total_records == (valid_records + rejected_records)

        MAX_REJECTION_RATE = 0.30
        if rejection_rate > MAX_REJECTION_RATE:
            logger.error(f"Rejection rate {rejection_rate:.2%} exceeded limit {MAX_REJECTION_RATE:.2%}")
            raise ValueError("Data quality threshold exceeded")

        transformed_df = (
            valid_df
            .withColumn("order_date", F.col("parsed_date"))
            .withColumn("revenue", F.col("quantity") * F.col("price"))
        )

        report_path = Path(config["log_path"]) / "pipeline_report.json"
        pipeline_report = {
            "pipeline": config["project_name"],
            "run_time": datetime.now().isoformat(),
            "records_received": total_records,
            "records_valid": valid_records,
            "records_rejected": rejected_records,
            "rejection_rate": builtins.round(rejection_rate, 4),
            "status": "SUCCESS"
        }

        with open(report_path, "w") as f:
            json.dump(pipeline_report, f, indent=4)

        logger.info("========== PIPELINE SUCCESS ==========")
        return pipeline_report

    except Exception as e:

        logger.exception(
            f"========== PIPELINE FAILED: {e} =========="
        )

        raise


### ⚙️ Step 1: Initialize SparkSession for Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing


In [276]:
import os
import json
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Day8_Production_Pipeline")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark started successfully")

Spark started successfully


In [277]:
CONFIG = {
    "source_path": "day7_project/source/orders.csv",
    "bronze_path": "day7_project/output/bronze/orders",
    "silver_path": "day7_project/output/silver/orders",
    "gold_path": "day7_project/output/gold/customer_report",
    "rejected_path": "day7_project/output/rejected/orders",
    "log_path": "day7_project/logs/pipeline.log",
    "report_path": "day7_project/output/pipeline_report.json"
}

In [278]:
for key, path in CONFIG.items():
    if key.endswith("_path"):
        if "." not in os.path.basename(path):
            os.makedirs(path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(path), exist_ok=True)
print("Directories ready")

Directories ready


In [279]:
logging.basicConfig(
    filename=CONFIG["log_path"],
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Day8Pipeline")
logger.info("Pipeline logging initialized")
print("Logging configured")

2026-10-08 07:43:38,091 - INFO - Pipeline logging initialized


Logging configured


In [280]:
def ingest_data(spark, config):
    try:
        logger.info("Starting data ingestion")
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(config["source_path"])
        )
        logger.info("Data ingestion successful")
        return df
    except Exception as e:
        logger.exception("Data ingestion failed")
        raise

In [281]:
def validate_data(df):
    logger.info("Starting data validation")
    validated = (
        df
        .withColumn(
            "parsed_order_date",
            F.try_to_date(F.col("order_date"), "yyyy-MM-dd")  # <-- Changed from to_date to try_to_date
        )
        .withColumn(
            "validation_error",
            F.concat_ws(
                ", ",
                F.when(F.col("order_id").isNull(), "missing_order_id"),
                F.when(
                    (F.col("customer_id").isNull()) | (F.col("customer_id") <= 0),
                    "invalid_customer_id"
                ),
                F.when(
                    (F.col("quantity").isNull()) | (F.col("quantity") <= 0),
                    "invalid_quantity"
                ),
                F.when(
                    (F.col("price").isNull()) | (F.col("price") < 100),
                    "invalid_price"
                ),
                F.when(
                    F.col("parsed_order_date").isNull(),
                    "invalid_order_date"
                )
            )
        )
    )
    
    valid_df = validated.filter(F.col("validation_error") == "")
    rejected_df = validated.filter(F.col("validation_error") != "")
    
    logger.info("Validation completed")
    return valid_df, rejected_df

In [282]:
import builtins  # Explicitly use Python's built-in round

def calculate_metrics(df, valid_df, rejected_df):
    total_count = df.count()
    valid_count = valid_df.count()
    rejected_count = rejected_df.count()
    
    rejection_rate = (
        (rejected_count / total_count * 100)
        if total_count > 0
        else 0
    )
    
    metrics = {
        "total_records": total_count,
        "valid_records": valid_count,
        "rejected_records": rejected_count,
        "rejection_rate": builtins.round(rejection_rate, 2)  # Use builtins.round
    }
    
    logger.info(f"Metrics: {metrics}")
    return metrics

### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [283]:
import builtins
import json
from datetime import datetime

def run_pipeline(spark, config):
    try:
        # 1. Log pipeline start
        logger.info("=== PIPELINE RUN STARTED ===")
        
        # 2. Ingest data
        df = ingest_data(spark, config)
        
        # 3. Validate data
        valid_df, rejected_df = validate_data(df)
        
        # 4. Calculate metrics
        metrics = calculate_metrics(df, valid_df, rejected_df)
        
        # 5. Check rejection threshold (e.g., max 30%)
        MAX_REJECTION_RATE = 30.0
        if metrics["rejection_rate"] > MAX_REJECTION_RATE:
            logger.error(
                f"Rejection rate {metrics['rejection_rate']}% "
                f"exceeded threshold {MAX_REJECTION_RATE}%"
            )
            raise ValueError("Data quality threshold exceeded. Pipeline stopped.")
        else:
            logger.info(
                f"Data quality passed: {metrics['rejection_rate']}% rejection rate"
            )
        
        # 6. Transform valid data
        logger.info("Starting transformations")
        transformed_df = (
            valid_df
            .withColumn("revenue", F.col("quantity") * F.col("price"))
            .withColumn("order_year", F.year("parsed_order_date"))
            .withColumn("order_month", F.month("parsed_order_date"))
        )
        
        # 7. Write Silver Data
        (
            transformed_df
            .write
            .mode("overwrite")
            .parquet(config["silver_path"])
        )
        logger.info("Silver data written successfully")
        
        # 8. Write Rejected Data
        (
            rejected_df
            .write
            .mode("overwrite")
            .parquet(config["rejected_path"])
        )
        logger.info("Rejected records written successfully")
        
        # 9. Create Gold Report (Customer-level aggregates)
        customer_report = (
            transformed_df
            .groupBy("customer_id")
            .agg(
                F.count("*").alias("total_orders"),
                F.sum("revenue").alias("total_revenue"),
                F.avg("revenue").alias("avg_order_value")
            )
            .orderBy(F.col("total_revenue").desc())
        )
        
        (
            customer_report
            .write
            .mode("overwrite")
            .parquet(config["gold_path"])
        )
        logger.info("Gold customer report written successfully")
        
        # 10. Create Pipeline JSON Report
        pipeline_report = {
            "pipeline": "Day8_Production_Pipeline",
            "run_time": datetime.now().isoformat(),
            "status": "SUCCESS",
            "metrics": metrics,
            "outputs": {
                "silver": config["silver_path"],
                "gold": config["gold_path"],
                "rejected": config["rejected_path"]
            }
        }
        
        with open(config["report_path"], "w") as f:
            json.dump(pipeline_report, f, indent=4)
        
        logger.info("Pipeline report generated successfully")
        logger.info("=== PIPELINE RUN COMPLETED SUCCESSFULLY ===")
        print("Pipeline executed successfully!")
        
        return transformed_df, metrics

    except Exception as e:
        logger.exception("Pipeline failed with exception")
        print(f"Pipeline failed: {e}")
        raise

# Execute the pipeline
transformed_df, metrics = run_pipeline(spark, CONFIG)

2026-10-08 07:43:38,188 - INFO - === PIPELINE RUN STARTED ===
2026-10-08 07:43:38,188 - INFO - Starting data ingestion
2026-10-08 07:43:38,368 - INFO - Data ingestion successful
2026-10-08 07:43:38,368 - INFO - Starting data validation
2026-10-08 07:43:38,477 - INFO - Validation completed
2026-10-08 07:43:38,858 - INFO - Metrics: {'total_records': 10, 'valid_records': 7, 'rejected_records': 3, 'rejection_rate': 30.0}
2026-10-08 07:43:38,867 - INFO - Data quality passed: 30.0% rejection rate
2026-10-08 07:43:38,867 - INFO - Starting transformations
2026-10-08 07:43:39,608 - INFO - Silver data written successfully
2026-10-08 07:43:40,338 - INFO - Rejected records written successfully
2026-10-08 07:43:41,148 - INFO - Gold customer report written successfully
2026-10-08 07:43:41,148 - INFO - Pipeline report generated successfully
2026-10-08 07:43:41,156 - INFO - === PIPELINE RUN COMPLETED SUCCESSFULLY ===


Pipeline executed successfully!


transformed_df, metrics = run_pipeline(spark, CONFIG)


### 🧪 Automated Production Unit Test Suite


In [285]:
print("\nRunning pipeline unit tests...")

# Test 1 — Reconciliation
assert metrics["total_records"] == metrics["valid_records"] + metrics["rejected_records"], \
    "Test 1 Failed: Record count reconciliation mismatch!"
print("Test 1 PASSED: Reconciliation matches")

# Test 2 — No invalid customer IDs in Silver layer
invalid_customers = transformed_df.filter(F.col("customer_id") <= 0).count()
assert invalid_customers == 0, "Test 2 Failed: Found invalid customer IDs in silver!"
print("Test 2 PASSED: No invalid customer IDs")

# Test 3 — No negative revenue
negative_revenue = transformed_df.filter(F.col("revenue") < 0).count()
assert negative_revenue == 0, "Test 3 Failed: Found negative revenue!"
print("Test 3 PASSED: No negative revenue")

# Test 4 — Revenue calculation check
revenue_errors = transformed_df.filter(
    F.col("revenue") != F.col("quantity") * F.col("price")
).count()
assert revenue_errors == 0, "Test 4 Failed: Revenue calculation error!"
print("Test 4 PASSED: Revenue calculations accurate")

print("\nAll tests passed successfully! Ready for Day 9.")


Running pipeline unit tests...
Test 1 PASSED: Reconciliation matches
Test 2 PASSED: No invalid customer IDs
Test 3 PASSED: No negative revenue
Test 4 PASSED: Revenue calculations accurate

All tests passed successfully! Ready for Day 9.


## ☁️ Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion
### 🎯 Learning Objectives:
1. Architect a **Hybrid Cloud Lakehouse**: Ingesting from operational cloud OLTP (Neon Serverless PostgreSQL) into a distributed analytics lake.
2. Configure Apache Spark with external PostgreSQL JDBC drivers (`postgresql-42.6.0.jar`).
3. Leverage dual ingestion patterns:
   - Direct JDBC driver streaming for large distributed tables.
   - `psycopg2` + Pandas to Spark DataFrame bridge for serverless environments.
4. Perform distributed relational joins and data cleansing on cloud-ingested datasets.
5. Export curated business data into local partitioned Parquet Gold lakehouse tables.
---


PostgreSQL
    │
    │ JDBC
    ▼
  PySpark
    │
    ├── Read
    ├── Transform
    ├── Filter
    └── Aggregate
    │
    ▼
Parquet


### ⚙️ Step 1: Initialize SparkSession for Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion


In [287]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Day9_Postgres_Pipeline")
    .master("local[*]")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark started with PostgreSQL JDBC support successfully!")

Spark started with PostgreSQL JDBC support successfully!


### ⚙️ Step 1: Initialize SparkSession for Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion


In [288]:
from pyspark.sql import SparkSession

# Initialize Spark Session with PostgreSQL JDBC driver package
spark = SparkSession.builder \
    .appName("Day9_Neon_Pipeline") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

print("Spark session successfully created with PostgreSQL JDBC driver package!")

Spark session successfully created with PostgreSQL JDBC driver package!


In [289]:
import os
# Replace these with your cloud PostgreSQL credentials (e.g., from Neon or Supabase)
DB_HOST = os.getenv("NEON_DB_HOST", "ep-your-subdomain.aws.neon.tech")  # e.g., ep-cool-cloud-123456.us-east-2.aws.neon.tech
DB_NAME = "neondb"                        # or de_project
DB_USER = "neondb_owner"                 # e.g., neondb_owner
DB_PASSWORD = os.getenv("NEON_DB_PASSWORD", "your_neon_password_here")             # your database password

# Direct URL format using Neon's connection string parameters
jdbc_url = "jdbc:postgresql://ep-your-subdomain.aws.neon.tech/neondb?sslmode=require"

connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

print("Configuration ready")

Configuration ready


!pip install psycopg2-binary


In [291]:
import psycopg2

# Connect to cloud PostgreSQL using psycopg2
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)
cursor = conn.cursor()

# 1. Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    product VARCHAR(100),
    quantity INT,
    price DECIMAL(10,2),
    order_date DATE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    signup_date DATE
);
""")

# 2. Insert Orders Data
cursor.execute("DELETE FROM orders;") # Clear existing to avoid duplicate key errors on re-run
cursor.execute("""
INSERT INTO orders VALUES
(1001, 101, 'Rahul', 'Kolkata', 'Laptop', 1, 65000, '2026-09-01'),
(1002, 102, 'Amit', 'Delhi', 'Mouse', 2, 800, '2026-09-02'),
(1003, 101, 'Rahul', 'Kolkata', 'Keyboard', 1, 2500, '2026-09-03'),
(1004, 103, 'Priya', 'Mumbai', 'Monitor', 2, 15000, '2026-09-04'),
(1005, 104, 'Sneha', 'Pune', 'Laptop', 1, 70000, '2026-09-05'),
(1006, 102, 'Amit', 'Delhi', 'Headphones', 2, 3000, '2026-09-06'),
(1007, 101, 'Rahul', 'Kolkata', 'Mouse', 3, 750, '2026-09-07'),
(1008, 103, 'Priya', 'Mumbai', 'Laptop', 1, 68000, '2026-09-08'),
(1009, 104, 'Sneha', 'Pune', 'Keyboard', 2, 2200, '2026-09-09'),
(1010, 102, 'Amit', 'Delhi', 'Monitor', 1, 14000, '2026-09-10');
""")

# 3. Insert Customers Data
cursor.execute("DELETE FROM customers;")
cursor.execute("""
INSERT INTO customers VALUES
(101, 'Rahul', 'Kolkata', '2026-08-01'),
(102, 'Amit', 'Delhi', '2026-08-05'),
(103, 'Priya', 'Mumbai', '2026-08-10'),
(104, 'Sneha', 'Pune', '2026-08-15');
""")

conn.commit()
cursor.close()
conn.close()
print("Tables created and practice data inserted successfully via Jupyter!")

Tables created and practice data inserted successfully via Jupyter!


### ⚙️ Step 1: Initialize SparkSession for Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion


In [292]:
import os
import psycopg2
import pandas as pd
from pyspark.sql import SparkSession

# 1. Ensure Spark session is active
if 'spark' not in locals() or not spark:
    spark = SparkSession.builder.appName("Day9_Neon_Pandas_Bridge").getOrCreate()

# 2. Your Neon PostgreSQL Credentials
DB_HOST = os.getenv("NEON_DB_HOST", "ep-your-subdomain.aws.neon.tech")
DB_NAME = "neondb"
DB_USER = "neondb_owner"
DB_PASSWORD = os.getenv("NEON_DB_PASSWORD", "your_neon_password_here")

print("Connecting to Neon PostgreSQL via psycopg2...")

# 3. Connect and query the table using psycopg2 & pandas
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)

# Read the 'orders' table directly into a Pandas DataFrame
pdf = pd.read_sql("SELECT * FROM orders;", con=conn)
conn.close()

print(f"Fetched {len(pdf)} rows into Pandas successfully!")

# 4. Convert the Pandas DataFrame into a PySpark DataFrame
orders_pg = spark.createDataFrame(pdf)

# 5. Verify the data in Spark
print(f"Total records read into PySpark: {orders_pg.count()}")
orders_pg.printSchema()
orders_pg.show(5)

Connecting to Neon PostgreSQL via psycopg2...


C:\Users\Me\AppData\Local\Temp\ipykernel_1312\700499037.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pdf = pd.read_sql("SELECT * FROM orders;", con=conn)


Fetched 10 rows into Pandas successfully!
Total records read into PySpark: 10
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+-------------+-------+--------+--------+-------+----------+
|order_id|customer_id|customer_name|   city| product|quantity|  price|order_date|
+--------+-----------+-------------+-------+--------+--------+-------+----------+
|    1001|        101|        Rahul|Kolkata|  Laptop|       1|65000.0|2026-09-01|
|    1002|        102|         Amit|  Delhi|   Mouse|       2|  800.0|2026-09-02|
|    1003|        101|        Rahul|Kolkata|Keyboard|       1| 2500.0|2026-09-03|
|    1004|        103|        Priya| Mumbai| Monitor|       2|15000.0|2026-09-04|
|    1005|        104|      

In [293]:
import pyspark.sql.functions as F

print("--- 1. Running Data Transformations on orders_pg ---")

# Example transformation: Clean/explore schema and calculate metrics
# (Adjust column names based on your actual table schema, e.g., 'total_amount', 'status', 'order_date')

# Show summary stats of the orders DataFrame
orders_pg.describe().show()

# Example Aggregation: Total orders and revenue by order status (if 'status' and 'amount' exist)
# Let's check available columns dynamically to prevent key errors:
columns = orders_pg.columns
print(f"Available columns in orders table: {columns}")

# Safe aggregation based on common column names
if "status" in columns and "total_amount" in columns:
    status_summary = (
        orders_pg.groupBy("status")
        .agg(
            F.count("order_id").alias("order_count"),
            F.sum("total_amount").alias("total_revenue"),
            F.avg("total_amount").alias("avg_order_value")
        )
        .orderBy(F.desc("total_revenue"))
    )
    print("Order Summary by Status:")
    status_summary.show()
else:
    # Fallback basic grouping on the first couple of columns
    print("Performing basic count grouping...")
    orders_pg.groupBy(columns[1] if len(columns) > 1 else columns[0]).count().show(10)

--- 1. Running Data Transformations on orders_pg ---
+-------+------------------+------------------+-------------+-----+----------+------------------+------------------+
|summary|          order_id|       customer_id|customer_name| city|   product|          quantity|             price|
+-------+------------------+------------------+-------------+-----+----------+------------------+------------------+
|  count|                10|                10|           10|   10|        10|                10|                10|
|   mean|            1005.5|             102.3|         NULL| NULL|      NULL|               1.6|           24125.0|
| stddev|3.0276503540974917|1.1595018087284072|         NULL| NULL|      NULL|0.6992058987801011|30497.971244002445|
|    min|              1001|               101|         Amit|Delhi|Headphones|                 1|             750.0|
|    max|              1010|               104|        Sneha| Pune|     Mouse|                 3|           70000.0|
+-------+--

In [294]:
print("--- 2. Exporting processed data ---")

# Option A: Save processed PySpark DataFrame to a local Parquet file for downstream tasks
output_path = "output_orders_processed.parquet"
orders_pg.write.mode("overwrite").parquet(output_path)
print(f"Successfully saved processed orders to Parquet at: {output_path}")

# Option B: If you want to push results back to Postgres using your psycopg2 connection:
# import psycopg2
# conn = psycopg2.connect(host=DB_HOST, database=DB_NAME, user=DB_USER, password=DB_PASSWORD, sslmode="require")
# cursor = conn.cursor()
# # write logic or execute inserts here...
# conn.close()

--- 2. Exporting processed data ---
Successfully saved processed orders to Parquet at: output_orders_processed.parquet


### ⚙️ Step 1: Initialize SparkSession for Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion


In [295]:
import os
import psycopg2
import pandas as pd
from pyspark.sql import SparkSession

# 1. Ensure Spark session is active
if 'spark' not in locals() or not spark:
    spark = SparkSession.builder.appName("Day9_Neon_Customer_Revenue").getOrCreate()

# 2. Neon PostgreSQL Credentials
DB_HOST = os.getenv("NEON_DB_HOST", "ep-your-subdomain.aws.neon.tech")
DB_NAME = "neondb"
DB_USER = "neondb_owner"
DB_PASSWORD = os.getenv("NEON_DB_PASSWORD", "your_neon_password_here")

# 3. Define the custom query
query = """
    SELECT 
        customer_id, 
        SUM(quantity * price) AS total_revenue 
    FROM orders 
    GROUP BY customer_id
    ORDER BY total_revenue DESC;
"""

print("Executing query on Neon PostgreSQL...")

# 4. Connect and fetch the query results into a Pandas DataFrame
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)

pdf = pd.read_sql(query, con=conn)
conn.close()

print(f"Query executed! Fetched {len(pdf)} rows.")

# 5. Convert the Pandas DataFrame into a PySpark DataFrame
revenue_df = spark.createDataFrame(pdf)

# 6. Show the top customer revenues
revenue_df.show(10, truncate=False)

Executing query on Neon PostgreSQL...


C:\Users\Me\AppData\Local\Temp\ipykernel_1312\2686383354.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pdf = pd.read_sql(query, con=conn)


Query executed! Fetched 4 rows.
+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|103        |98000.0      |
|104        |74400.0      |
|101        |69750.0      |
|102        |21600.0      |
+-----------+-------------+



#         PostgreSQL
     #         ┌─────────┐
     #         │ orders  │
     #         │customers│
     #         └────┬────┘
     #              │
     #             JDBC
     #              │
     #              ▼
     #          PySpark
     #              │
     #      ┌───────┴────────┐
     #      ▼                ▼
     # Transform           Join
     #      │                │
     #      └───────┬────────┘
     #              ▼
     #          Aggregate
     #              │
     #              ▼
     #         Gold Parquet


## 🎯 Day 10: Custom DAG Pipeline Orchestrator with Retries & Fault Tolerance
### 🎯 Learning Objectives:
1. Build an Airflow-style **Directed Acyclic Graph (DAG) Pipeline Orchestrator** from scratch in Python (`SimpleOrchestrator`).
2. Implement core workflow management capabilities:
   - Task registration and sequential dependency resolution.
   - Shared execution context dictionary for inter-task data passing.
   - Automatic retries with customizable retry counts and delay backoff.
   - Comprehensive task state machine: `PENDING` -> `RUNNING` -> `SUCCESS` / `FAILED`.
3. Conduct **Fault Injection & Chaos Testing**:
   - Verify that transient errors trigger retries.
   - Verify that persistent failures halt downstream execution safely.
4. Understand architectural boundaries: PySpark as the **Compute Engine** vs. Airflow/DAGs as the **Orchestration Layer**.
---


In [297]:
#DAY 10
# Step 1: Define Your Pipeline Tasks (Cell 1)
import logging
import time

# Configure logging to mimic Airflow's task logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def extract():
    logging.info("--- TASK START: Extract ---")
    logging.info("Reading data from Neon PostgreSQL (or CSV)...")
    # Simulate data extraction
    data = [{"id": 1, "item": "Laptop", "price": 1000}, {"id": 2, "item": "Mouse", "price": 50}]
    logging.info("Extract completed successfully.")
    return data

def validate(data):
    logging.info("--- TASK START: Validate ---")
    if not data:
        raise ValueError("Validation failed: Dataset is empty!")
    logging.info(f"Validated {len(data)} records successfully.")
    return data

def transform(data):
    logging.info("--- TASK START: Transform ---")
    # Let's add total revenue calculation
    for row in data:
        row["total_value"] = row["price"] * 1.18  # adding tax
    logging.info("Transformation completed successfully.")
    return data

def load(data):
    logging.info("--- TASK START: Load ---")
    logging.info(f"Loading {len(data)} records into target destination...")
    logging.info("Load completed successfully.")

def quality_check():
    logging.info("--- TASK START: Quality Check ---")
    logging.info("Running post-load data checks... All constraints passed!")

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [298]:
# Step 2: Build the Orchestrator Engine with Retries & Failure Handling (Cell 2)
class SimpleOrchestrator:
    def __init__(self, dag_name, max_retries=2):
        self.dag_name = dag_name
        self.max_retries = max_retries
        self.tasks = []

    def add_task(self, name, func, *args):
        self.tasks.append({"name": name, "func": func, "args": args})

    def run(self):
        logging.info(f"=== PIPELINE START: {self.dag_name} ===")
        context_data = None

        for task in self.tasks:
            task_name = task["name"]
            func = task["func"]
            args = task["args"]
            
            success = False
            attempt = 0

            while attempt <= self.max_retries and not success:
                try:
                    if attempt > 0:
                        logging.warning(f"RETRY {attempt}/{self.max_retries} for task: {task_name}")
                    
                    # Execute task (passing context data if it exists from previous task)
                    if args:
                        context_data = func(*args)
                    elif context_data is not None:
                        context_data = func(context_data)
                    else:
                        context_data = func()
                        
                    success = True
                    logging.info(f"Task '{task_name}' status: SUCCESS ✅")
                    
                except Exception as e:
                    attempt += 1
                    logging.error(f"Task '{task_name}' failed with error: {e}")
                    if attempt > self.max_retries:
                        logging.critical(f"Task '{task_name}' exceeded max retries. Pipeline HALTED ❌")
                        logging.info(f"=== PIPELINE FAILED: {self.dag_name} ===")
                        return False
                        
        logging.info(f"=== PIPELINE SUCCESS: {self.dag_name} ===")
        return True

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [299]:
# Step 3: Run and Test Your Pipeline (Cell 3)
# Initialize your pipeline DAG
pipeline = SimpleOrchestrator(dag_name="day10_ecommerce_pipeline", max_retries=2)

# Register tasks in order (Dependencies)
pipeline.add_task("extract", extract)
pipeline.add_task("validate", validate) # takes output of extract automatically
pipeline.add_task("transform", transform) # takes output of validate automatically
pipeline.add_task("load", load)
pipeline.add_task("quality_check", quality_check)

# Execute the orchestrated pipeline
pipeline.run()

2026-10-08 07:45:19,511 - INFO - === PIPELINE START: day10_ecommerce_pipeline ===
2026-10-08 07:45:19,512 - INFO - --- TASK START: Extract ---
2026-10-08 07:45:19,513 - INFO - Reading data from Neon PostgreSQL (or CSV)...
2026-10-08 07:45:19,514 - INFO - Extract completed successfully.
2026-10-08 07:45:19,515 - INFO - Task 'extract' status: SUCCESS ✅
2026-10-08 07:45:19,515 - INFO - --- TASK START: Validate ---
2026-10-08 07:45:19,516 - INFO - Validated 2 records successfully.
2026-10-08 07:45:19,516 - INFO - Task 'validate' status: SUCCESS ✅
2026-10-08 07:45:19,517 - INFO - --- TASK START: Transform ---
2026-10-08 07:45:19,518 - INFO - Transformation completed successfully.
2026-10-08 07:45:19,519 - INFO - Task 'transform' status: SUCCESS ✅
2026-10-08 07:45:19,519 - INFO - --- TASK START: Load ---
2026-10-08 07:45:19,520 - INFO - Loading 2 records into target destination...
2026-10-08 07:45:19,521 - INFO - Load completed successfully.
2026-10-08 07:45:19,521 - INFO - Task 'load' statu

True

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [300]:
# 🧪 Test the Failure & Retry Logic (Cell 4)
# Let's create a broken transform function to test retries and failure halt
def broken_transform(data):
    logging.info("--- TASK START: Transform (Broken) ---")
    raise Exception("Database connection timeout during transformation!")

# Build a failing pipeline
failing_pipeline = SimpleOrchestrator(dag_name="day10_failing_pipeline", max_retries=2)

failing_pipeline.add_task("extract", extract)
failing_pipeline.add_task("validate", validate)
failing_pipeline.add_task("transform", broken_transform) # This will fail and retry 2 times
failing_pipeline.add_task("load", load) # Should be skipped because transform fails

failing_pipeline.run()

2026-10-08 07:45:43,519 - INFO - === PIPELINE START: day10_failing_pipeline ===
2026-10-08 07:45:43,520 - INFO - --- TASK START: Extract ---
2026-10-08 07:45:43,520 - INFO - Reading data from Neon PostgreSQL (or CSV)...
2026-10-08 07:45:43,522 - INFO - Extract completed successfully.
2026-10-08 07:45:43,522 - INFO - Task 'extract' status: SUCCESS ✅
2026-10-08 07:45:43,524 - INFO - --- TASK START: Validate ---
2026-10-08 07:45:43,525 - INFO - Validated 2 records successfully.
2026-10-08 07:45:43,525 - INFO - Task 'validate' status: SUCCESS ✅
2026-10-08 07:45:43,526 - INFO - --- TASK START: Transform (Broken) ---
2026-10-08 07:45:43,526 - ERROR - Task 'transform' failed with error: Database connection timeout during transformation!
2026-10-08 07:45:43,527 - WARNING - RETRY 1/2 for task: transform
2026-10-08 07:45:43,528 - INFO - --- TASK START: Transform (Broken) ---
2026-10-08 07:45:43,529 - ERROR - Task 'transform' failed with error: Database connection timeout during transformation!
2

False

PySpark
   ↓
Processes/transforms data

Airflow
   ↓
Orchestrates when/how pipeline tasks run
